In [ ]:
!rm -rf /content/drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
!pip install -q tokenizers transformers datasets accelerate

In [ ]:
import os

CORPUS_PATH = "/content/drive/MyDrive/MUSE_artifacts/en_w2w_wiki_80M_v2_clean.txt"

print("File exists:", os.path.exists(CORPUS_PATH))

file_size_mb = os.path.getsize(CORPUS_PATH) / (1024 * 1024)
print("Corpus size (MB):", round(file_size_mb, 2))

In [ ]:
word_count = 0

with open(CORPUS_PATH, "r", encoding="utf-8") as f:
    for line in f:
        words = line.strip().split()
        word_count += len(words)

print("Total words:", word_count)

Train ByteLevelBPE Tokenizer

In [ ]:
from tokenizers import ByteLevelBPETokenizer
import os

TOKENIZER_DIR = "/content/drive/MyDrive/MUSE_artifacts/babyroberta_tokenizer"
os.makedirs(TOKENIZER_DIR, exist_ok=True)

tokenizer = ByteLevelBPETokenizer()

tokenizer.train(
    files=[CORPUS_PATH],
    vocab_size=32000,
    min_frequency=2,
    special_tokens=[
        "<s>",
        "<pad>",
        "</s>",
        "<unk>",
        "<mask>",
    ],
)

tokenizer.save_model(TOKENIZER_DIR)

print("Tokenizer saved to:", TOKENIZER_DIR)

Load as RoBERTa Tokenizer & Test

In [ ]:
from tokenizers import Tokenizer
from tokenizers.models import BPE
from tokenizers.trainers import BpeTrainer
from tokenizers.pre_tokenizers import ByteLevel
from tokenizers.processors import RobertaProcessing
from tokenizers.decoders import ByteLevel as ByteLevelDecoder
import os

TOKENIZER_DIR = "/content/drive/MyDrive/MUSE_artifacts/babyroberta_tokenizer"
os.makedirs(TOKENIZER_DIR, exist_ok=True)

# Initialize empty tokenizer
tokenizer = Tokenizer(BPE(unk_token="<unk>"))

tokenizer.pre_tokenizer = ByteLevel()

trainer = BpeTrainer(
    vocab_size=32000,
    min_frequency=2,
    special_tokens=["<s>", "<pad>", "</s>", "<unk>", "<mask>"],
)

print("Training tokenizer...")
tokenizer.train([CORPUS_PATH], trainer)

# Add RoBERTa post-processing
tokenizer.post_processor = RobertaProcessing(
    ("</s>", tokenizer.token_to_id("</s>")),
    ("<s>", tokenizer.token_to_id("<s>")),
)

tokenizer.decoder = ByteLevelDecoder()

# Save single tokenizer.json
tokenizer.save(os.path.join(TOKENIZER_DIR, "tokenizer.json"))

print("Tokenizer fully saved as tokenizer.json")

In [ ]:
from transformers import RobertaTokenizerFast

TOKENIZER_DIR = "/content/drive/MyDrive/MUSE_artifacts/babyroberta_tokenizer"

tokenizer = RobertaTokenizerFast(
    tokenizer_file=f"{TOKENIZER_DIR}/tokenizer.json",
    max_len=512
)

print("Tokenizer loaded successfully.")

sample = "This is a test sentence for BabyRoberta."
encoded = tokenizer(sample)

print("Encoded:", encoded["input_ids"][:20])
print("Decoded:", tokenizer.decode(encoded["input_ids"]))

In [ ]:
total_tokens = 0
sample_lines = 100000  # sample only

with open(CORPUS_PATH, "r", encoding="utf-8") as f:
    for i, line in enumerate(f):
        tokens = tokenizer(line.strip(), add_special_tokens=False)["input_ids"]
        total_tokens += len(tokens)

        if i >= sample_lines:
            break

avg_tokens_per_line = total_tokens / sample_lines

# now estimate total lines
with open(CORPUS_PATH, "r", encoding="utf-8") as f:
    total_lines = sum(1 for _ in f)

estimated_tokens = avg_tokens_per_line * total_lines

print("Estimated total tokens:", int(estimated_tokens))

Create BabyRoBERTa Config & Model

In [ ]:
from transformers import RobertaConfig, RobertaForMaskedLM

config = RobertaConfig(
    vocab_size=32000,
    max_position_embeddings=514,
    num_attention_heads=8,
    num_hidden_layers=6,
    type_vocab_size=1,
    hidden_size=512,
    intermediate_size=2048,
)

model = RobertaForMaskedLM(config)

print("Model created.")
print("Total parameters:", sum(p.numel() for p in model.parameters()) / 1e6, "Million")

Load Dataset (Streaming)

In [ ]:
from datasets import load_dataset

dataset = load_dataset(
    "text",
    data_files={"train": CORPUS_PATH},
    streaming=False
)

print(dataset)

In [ ]:
from transformers import DataCollatorForLanguageModeling

def tokenize_function(examples):
    return tokenizer(
        examples["text"],
        truncation=True,
        max_length=512,
        return_special_tokens_mask=True,
    )

tokenized_dataset = dataset.map(
    tokenize_function,
    batched=True,
    remove_columns=["text"],
    num_proc=2
)

print(tokenized_dataset)

Group Into 512-Token Blocks

In [ ]:
block_size = 512

def group_texts(examples):
    # Concatenate all texts
    concatenated = {}
    for key in examples.keys():
        concatenated[key] = sum(examples[key], [])

    total_length = len(concatenated["input_ids"])

    # Drop remainder
    total_length = (total_length // block_size) * block_size

    result = {}
    for key in concatenated.keys():
        result[key] = [
            concatenated[key][i : i + block_size]
            for i in range(0, total_length, block_size)
        ]

    return result

lm_dataset = tokenized_dataset.map(
    group_texts,
    batched=True,
    num_proc=2,
)

print(lm_dataset)

Setup MLM Data Collator

In [ ]:
from transformers import DataCollatorForLanguageModeling

data_collator = DataCollatorForLanguageModeling(
    tokenizer=tokenizer,
    mlm=True,
    mlm_probability=0.15,
)

print("MLM collator ready.")

Define Training Arguments

In [ ]:
from transformers import TrainingArguments

training_args = TrainingArguments(
    output_dir="/content/drive/MyDrive/MUSE_artifacts/babyroberta_model",
    num_train_epochs=5,
    per_device_train_batch_size=16,
    save_steps=2000,
    save_total_limit=2,
    prediction_loss_only=True,
    learning_rate=5e-4,
    weight_decay=0.01,
    warmup_steps=1000,
    logging_steps=500,
    fp16=True,
    report_to="none",
)

print("Training arguments set.")

In [ ]:
from transformers import Trainer

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=lm_dataset["train"],
    data_collator=data_collator,
)

print("Trainer ready.")

In [ ]:
trainer.train()

In [ ]:
# 🔹 choose a clear name (VERY IMPORTANT)
model_name = "babyroberta_30M_randomseed"

FINAL_DIR = f"/content/drive/MyDrive/MUSE_artifacts/{model_name}"

# 🔹 save model
trainer.save_model(FINAL_DIR)

# 🔹 save tokenizer (must be same folder)
tokenizer.save_pretrained(FINAL_DIR)

print("✅ Model saved at:", FINAL_DIR)

In [ ]:
import os

print("Files in saved folder:")
print(os.listdir(FINAL_DIR))

In [ ]:
import shutil

backup_dir = FINAL_DIR + "_backup"
shutil.copytree(FINAL_DIR, backup_dir)

print("✅ Backup created at:", backup_dir)

In [ ]:
from transformers import RobertaForMaskedLM, RobertaTokenizerFast
import torch

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

tokenizer = RobertaTokenizerFast.from_pretrained(FINAL_DIR)
model = RobertaForMaskedLM.from_pretrained(FINAL_DIR).to(device)

model.eval()

print("✅ Model reload successful")

In [ ]:
import os

BASE_DIR = "/content/drive/MyDrive/MUSE_artifacts/babyroberta_model"

for root, dirs, files in os.walk(BASE_DIR):
    print("\nFolder:", root)
    for f in files:
        print(" -", f)

In [ ]:
from transformers import TrainingArguments

training_args = TrainingArguments(
    output_dir="/content/drive/MyDrive/MUSE_artifacts/babyroberta_model",
    num_train_epochs=5,
    per_device_train_batch_size=16,
    save_steps=2000,
    save_total_limit=2,
    prediction_loss_only=True,
    learning_rate=5e-4,
    weight_decay=0.01,
    warmup_steps=1000,
    logging_steps=500,
    fp16=True,
    report_to="none",
)

In [ ]:
from transformers import Trainer

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=lm_dataset["train"],
    data_collator=data_collator,
)

trainer.train(resume_from_checkpoint="/content/drive/MyDrive/MUSE_artifacts/babyroberta_model/checkpoint-14253")

In [ ]:
FINAL_DIR = "/content/drive/MyDrive/MUSE_artifacts/babyroberta_final"

trainer.save_model(FINAL_DIR)
tokenizer.save_pretrained(FINAL_DIR)

print("Final model saved to:", FINAL_DIR)

In [ ]:
from transformers import RobertaTokenizerFast, RobertaForMaskedLM
import torch

MODEL_DIR = "/content/drive/MyDrive/MUSE_artifacts/babyroberta_final"

tokenizer = RobertaTokenizerFast.from_pretrained(MODEL_DIR)
model = RobertaForMaskedLM.from_pretrained(MODEL_DIR)

model.eval()

print("Model + tokenizer loaded successfully.")

 Test Masked Word Prediction

In [ ]:
import torch

text = "The company will <mask> a new product next year."

inputs = tokenizer(text, return_tensors="pt")
mask_token_index = torch.where(inputs["input_ids"] == tokenizer.mask_token_id)[1]

with torch.no_grad():
    outputs = model(**inputs)
    logits = outputs.logits

mask_token_logits = logits[0, mask_token_index, :]
top_tokens = torch.topk(mask_token_logits, 5, dim=1).indices[0].tolist()

print("Input:", text)
print("\nTop 5 predictions:")

for token in top_tokens:
    print(tokenizer.decode([token]))

Pseudo-Perplexity Function

In [ ]:
import torch
import math
from tqdm import tqdm

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)

def compute_pseudo_perplexity(text, tokenizer, model, max_length=512):
    model.eval()

    encodings = tokenizer(
        text,
        return_tensors="pt",
        truncation=True,
        max_length=max_length
    )

    input_ids = encodings["input_ids"].to(device)
    attention_mask = encodings["attention_mask"].to(device)

    seq_len = input_ids.size(1)
    log_likelihood = 0.0
    token_count = 0

    with torch.no_grad():
        for i in range(1, seq_len - 1):  # skip <s> and </s>
            masked_input = input_ids.clone()
            masked_input[0, i] = tokenizer.mask_token_id

            outputs = model(masked_input, attention_mask=attention_mask)
            logits = outputs.logits

            log_probs = torch.log_softmax(logits[0, i], dim=-1)
            true_token = input_ids[0, i]

            log_likelihood += log_probs[true_token].item()
            token_count += 1

    ppl = math.exp(-log_likelihood / token_count)
    return ppl

In [ ]:
with open("/content/drive/MyDrive/SLA_Project/chinese_icnale_clean.txt", "r", encoding="utf-8") as f:
    raw_text = f.read()

print("Length of text:", len(raw_text))
print("\nDoes it contain double newlines?", "\n\n" in raw_text)

In [ ]:
import re

# show first 1000 characters
print(raw_text[:1000])

# try to find common essay starters
starts = re.findall(r"(I agree|I disagree|In my opinion)", raw_text)
print("\nDetected starter count:", len(starts))

In [ ]:
import re

# Split while keeping the starter phrases
essays = re.split(r'(?=I agree|I disagree|In my opinion)', raw_text)

# Remove empty fragments
essays = [e.strip() for e in essays if len(e.strip()) > 100]

print("Number of reconstructed essays:", len(essays))
print("\nFirst essay preview:\n")
print(essays[0][:400])

In [ ]:
import numpy as np
from tqdm import tqdm

print("Re-evaluating Chinese_ICNALE properly...")

ppl_scores = []

for essay in tqdm(essays[:200]):  # use first 200 for fairness
    ppl = compute_pseudo_perplexity(essay, tokenizer, model)
    ppl_scores.append(ppl)

avg_ppl = np.mean(ppl_scores)

print("\nChinese_ICNALE Average PPL (fixed):", round(avg_ppl, 3))

In [ ]:
native_path = "/content/drive/MyDrive/SLA_Project/native_wikitext.txt"

In [ ]:
with open(native_path, "r", encoding="utf-8") as f:
    native_lines = f.readlines()

print("Number of lines:", len(native_lines))
print("\nPreview:\n")
print(native_lines[0][:500])

In [ ]:
import numpy as np
from tqdm import tqdm

print("Evaluating Native English...")

native_ppl_scores = []

for line in tqdm(native_lines[:200]):
    line = line.strip()
    if len(line.split()) < 5:
        continue

    ppl = compute_pseudo_perplexity(line, tokenizer, model)
    native_ppl_scores.append(ppl)

native_avg = np.mean(native_ppl_scores)

print("\nNative English Average PPL:", round(native_avg, 3))

In [ ]:
import numpy as np

essay_files = {
    "Japanese_ICNALE": "/content/drive/MyDrive/SLA_Project/japanese_icnale_clean.txt",
    "Korean_ICNALE": "/content/drive/MyDrive/SLA_Project/kor_icnale_clean.txt",
    "Thai_ICNALE": "/content/drive/MyDrive/SLA_Project/tha_icnale_clean.txt",
    "Filipino_ICNALE": "/content/drive/MyDrive/SLA_Project/phl_icnale_clean.txt",
    "Urdu_ICNALE": "/content/drive/MyDrive/SLA_Project/urdu_icnale_clean.txt",
}

results = {}

for group, path in essay_files.items():
    print(f"\nEvaluating {group}...")

    ppl_scores = []

    with open(path, "r", encoding="utf-8") as f:
        essays = f.readlines()

    # limit first 200 essays for speed (adjust later)
    essays = essays[:200]

    for essay in tqdm(essays):
        essay = essay.strip()
        if len(essay.split()) < 5:
            continue

        ppl = compute_pseudo_perplexity(essay, tokenizer, model)
        ppl_scores.append(ppl)

    avg_ppl = np.mean(ppl_scores)
    results[group] = avg_ppl

    print(f"{group} Average PPL: {avg_ppl:.3f}")

print("\nFinal Results:")
for k, v in results.items():
    print(k, "->", round(v, 3))

eed = 42, using

In [ ]:
# ==============================
# Train 30M BabyRoBERTa (Seed 42)
# ==============================

import os
import random
import numpy as np
import torch
from google.colab import drive
from datasets import load_dataset
from transformers import (
    RobertaTokenizerFast,
    RobertaConfig,
    RobertaForMaskedLM,
    DataCollatorForLanguageModeling,
    TrainingArguments,
    Trainer,
    set_seed
)

# Mount Drive
drive.mount('/content/drive')

# ------------------------------
# Set Seed
# ------------------------------
SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
set_seed(SEED)

# ------------------------------
# Paths
# ------------------------------
CORPUS_PATH = "/content/drive/MyDrive/MUSE_artifacts/en_w2w_wiki_80M_v2_clean.txt"
TOKENIZER_DIR = "/content/drive/MyDrive/MUSE_artifacts/babyroberta_tokenizer"
OUTPUT_DIR = "/content/drive/MyDrive/MUSE_artifacts/babyroberta_30M_seed42"
FINAL_DIR = "/content/drive/MyDrive/MUSE_artifacts/babyroberta_30M_seed42_final"

# ------------------------------
# Load Tokenizer
# ------------------------------
tokenizer = RobertaTokenizerFast(
    tokenizer_file=f"{TOKENIZER_DIR}/tokenizer.json",
    max_len=512
)

# ------------------------------
# Create Model (Same Architecture)
# ------------------------------
config = RobertaConfig(
    vocab_size=32000,
    max_position_embeddings=514,
    num_attention_heads=8,
    num_hidden_layers=6,
    type_vocab_size=1,
    hidden_size=512,
    intermediate_size=2048,
)

model = RobertaForMaskedLM(config)

print("Model created with seed:", SEED)
print("Total parameters:", sum(p.numel() for p in model.parameters()) / 1e6, "Million")

# ------------------------------
# Load Dataset
# ------------------------------
dataset = load_dataset(
    "text",
    data_files={"train": CORPUS_PATH},
    streaming=False
)

def tokenize_function(examples):
    return tokenizer(
        examples["text"],
        truncation=True,
        max_length=512,
        return_special_tokens_mask=True,
    )

tokenized_dataset = dataset.map(
    tokenize_function,
    batched=True,
    remove_columns=["text"],
    num_proc=2
)

# ------------------------------
# Group into 512 token blocks
# ------------------------------
block_size = 512

def group_texts(examples):
    concatenated = {}
    for key in examples.keys():
        concatenated[key] = sum(examples[key], [])

    total_length = len(concatenated["input_ids"])
    total_length = (total_length // block_size) * block_size

    result = {}
    for key in concatenated.keys():
        result[key] = [
            concatenated[key][i : i + block_size]
            for i in range(0, total_length, block_size)
        ]

    return result

lm_dataset = tokenized_dataset.map(
    group_texts,
    batched=True,
    num_proc=2,
)

# ------------------------------
# MLM Data Collator
# ------------------------------
data_collator = DataCollatorForLanguageModeling(
    tokenizer=tokenizer,
    mlm=True,
    mlm_probability=0.15,
)

# ------------------------------
# Training Arguments
# ------------------------------
training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    num_train_epochs=5,
    per_device_train_batch_size=16,
    save_steps=2000,
    save_total_limit=2,
    prediction_loss_only=True,
    learning_rate=5e-4,
    weight_decay=0.01,
    warmup_steps=1000,
    logging_steps=500,
    fp16=True,
    report_to="none",
    seed=SEED,
)

# ------------------------------
# Trainer
# ------------------------------
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=lm_dataset["train"],
    data_collator=data_collator,
)

# ------------------------------
# Train (Fresh — No Resume)
# ------------------------------
trainer.train()

# ------------------------------
# Save Final Model
# ------------------------------
trainer.save_model(FINAL_DIR)
tokenizer.save_pretrained(FINAL_DIR)

print("Training complete.")
print("Model saved to:", FINAL_DIR)

In [ ]:
import os

path = "/content/drive/MyDrive/MUSE_artifacts/babyroberta_30M_seed42_final"

print("Exists:", os.path.exists(path))
print("Files:", os.listdir(path))

In [ ]:
# ==============================
# Train 30M BabyRoBERTa (Seed 123)
# ==============================

import os
import random
import numpy as np
import torch
from google.colab import drive
from datasets import load_dataset
from transformers import (
    RobertaTokenizerFast,
    RobertaConfig,
    RobertaForMaskedLM,
    DataCollatorForLanguageModeling,
    TrainingArguments,
    Trainer,
    set_seed
)

# Mount Drive
drive.mount('/content/drive')

# ------------------------------
# Set Seed
# ------------------------------
SEED = 123

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
set_seed(SEED)

# ------------------------------
# Paths
# ------------------------------
CORPUS_PATH = "/content/drive/MyDrive/MUSE_artifacts/en_w2w_wiki_80M_v2_clean.txt"
TOKENIZER_DIR = "/content/drive/MyDrive/MUSE_artifacts/babyroberta_tokenizer"
OUTPUT_DIR = "/content/drive/MyDrive/MUSE_artifacts/babyroberta_30M_seed123"
FINAL_DIR = "/content/drive/MyDrive/MUSE_artifacts/babyroberta_30M_seed123_final"

# ------------------------------
# Load Tokenizer
# ------------------------------
tokenizer = RobertaTokenizerFast(
    tokenizer_file=f"{TOKENIZER_DIR}/tokenizer.json",
    max_len=512
)

# ------------------------------
# Create Model (Same Architecture)
# ------------------------------
config = RobertaConfig(
    vocab_size=32000,
    max_position_embeddings=514,
    num_attention_heads=8,
    num_hidden_layers=6,
    type_vocab_size=1,
    hidden_size=512,
    intermediate_size=2048,
)

model = RobertaForMaskedLM(config)

print("Model created with seed:", SEED)
print("Total parameters:", sum(p.numel() for p in model.parameters()) / 1e6, "Million")

# ------------------------------
# Load Dataset
# ------------------------------
dataset = load_dataset(
    "text",
    data_files={"train": CORPUS_PATH},
    streaming=False
)

def tokenize_function(examples):
    return tokenizer(
        examples["text"],
        truncation=True,
        max_length=512,
        return_special_tokens_mask=True,
    )

tokenized_dataset = dataset.map(
    tokenize_function,
    batched=True,
    remove_columns=["text"],
    num_proc=2
)

# ------------------------------
# Group into 512 token blocks
# ------------------------------
block_size = 512

def group_texts(examples):
    concatenated = {}
    for key in examples.keys():
        concatenated[key] = sum(examples[key], [])

    total_length = len(concatenated["input_ids"])
    total_length = (total_length // block_size) * block_size

    result = {}
    for key in concatenated.keys():
        result[key] = [
            concatenated[key][i : i + block_size]
            for i in range(0, total_length, block_size)
        ]

    return result

lm_dataset = tokenized_dataset.map(
    group_texts,
    batched=True,
    num_proc=2,
)

# ------------------------------
# MLM Data Collator
# ------------------------------
data_collator = DataCollatorForLanguageModeling(
    tokenizer=tokenizer,
    mlm=True,
    mlm_probability=0.15,
)

# ------------------------------
# Training Arguments
# ------------------------------
training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    num_train_epochs=5,
    per_device_train_batch_size=16,
    save_steps=2000,
    save_total_limit=2,
    prediction_loss_only=True,
    learning_rate=5e-4,
    weight_decay=0.01,
    warmup_steps=1000,
    logging_steps=500,
    fp16=True,
    report_to="none",
    seed=SEED,
)

# ------------------------------
# Trainer
# ------------------------------
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=lm_dataset["train"],
    data_collator=data_collator,
)

# ------------------------------
# Train (Fresh — No Resume)
# ------------------------------
trainer.train()

# ------------------------------
# Save Final Model
# ------------------------------
trainer.save_model(FINAL_DIR)
tokenizer.save_pretrained(FINAL_DIR)

print("Training complete.")
print("Model saved to:", FINAL_DIR)

In [ ]:
import os

path = "/content/drive/MyDrive/MUSE_artifacts/babyroberta_30M_seed123_final"

print("Exists:", os.path.exists(path))
print("Files:", os.listdir(path))

In [ ]:
# ==============================
# Train 30M BabyRoBERTa (Seed 999)
# ==============================

import os
import random
import numpy as np
import torch
from google.colab import drive
from datasets import load_dataset
from transformers import (
    RobertaTokenizerFast,
    RobertaConfig,
    RobertaForMaskedLM,
    DataCollatorForLanguageModeling,
    TrainingArguments,
    Trainer,
    set_seed
)

# Mount Drive
drive.mount('/content/drive')

# ------------------------------
# Set Seed
# ------------------------------
SEED = 999

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
set_seed(SEED)

# ------------------------------
# Paths
# ------------------------------
CORPUS_PATH = "/content/drive/MyDrive/MUSE_artifacts/en_w2w_wiki_80M_v2_clean.txt"
TOKENIZER_DIR = "/content/drive/MyDrive/MUSE_artifacts/babyroberta_tokenizer"
OUTPUT_DIR = "/content/drive/MyDrive/MUSE_artifacts/babyroberta_30M_seed999"
FINAL_DIR = "/content/drive/MyDrive/MUSE_artifacts/babyroberta_30M_seed999_final"

# ------------------------------
# Load Tokenizer
# ------------------------------
tokenizer = RobertaTokenizerFast(
    tokenizer_file=f"{TOKENIZER_DIR}/tokenizer.json",
    max_len=512
)

# ------------------------------
# Create Model (Same Architecture)
# ------------------------------
config = RobertaConfig(
    vocab_size=32000,
    max_position_embeddings=514,
    num_attention_heads=8,
    num_hidden_layers=6,
    type_vocab_size=1,
    hidden_size=512,
    intermediate_size=2048,
)

model = RobertaForMaskedLM(config)

print("Model created with seed:", SEED)
print("Total parameters:", sum(p.numel() for p in model.parameters()) / 1e6, "Million")

# ------------------------------
# Load Dataset
# ------------------------------
dataset = load_dataset(
    "text",
    data_files={"train": CORPUS_PATH},
    streaming=False
)

def tokenize_function(examples):
    return tokenizer(
        examples["text"],
        truncation=True,
        max_length=512,
        return_special_tokens_mask=True,
    )

tokenized_dataset = dataset.map(
    tokenize_function,
    batched=True,
    remove_columns=["text"],
    num_proc=2
)

# ------------------------------
# Group into 512 token blocks
# ------------------------------
block_size = 512

def group_texts(examples):
    concatenated = {}
    for key in examples.keys():
        concatenated[key] = sum(examples[key], [])

    total_length = len(concatenated["input_ids"])
    total_length = (total_length // block_size) * block_size

    result = {}
    for key in concatenated.keys():
        result[key] = [
            concatenated[key][i : i + block_size]
            for i in range(0, total_length, block_size)
        ]

    return result

lm_dataset = tokenized_dataset.map(
    group_texts,
    batched=True,
    num_proc=2,
)

# ------------------------------
# MLM Data Collator
# ------------------------------
data_collator = DataCollatorForLanguageModeling(
    tokenizer=tokenizer,
    mlm=True,
    mlm_probability=0.15,
)

# ------------------------------
# Training Arguments
# ------------------------------
training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    num_train_epochs=5,
    per_device_train_batch_size=16,
    save_steps=2000,
    save_total_limit=2,
    prediction_loss_only=True,
    learning_rate=5e-4,
    weight_decay=0.01,
    warmup_steps=1000,
    logging_steps=500,
    fp16=True,
    report_to="none",
    seed=SEED,
)

# ------------------------------
# Trainer
# ------------------------------
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=lm_dataset["train"],
    data_collator=data_collator,
)

# ------------------------------
# Train (Fresh — No Resume)
# ------------------------------
trainer.train()

# ------------------------------
# Save Final Model
# ------------------------------
trainer.save_model(FINAL_DIR)
tokenizer.save_pretrained(FINAL_DIR)

print("Training complete.")
print("Model saved to:", FINAL_DIR)

In [ ]:
import os

path = "/content/drive/MyDrive/MUSE_artifacts/babyroberta_30M_seed999_final"

print("Exists:", os.path.exists(path))
print("Files:", os.listdir(path))

In [ ]:
# ============================================
# Evaluate 30M Models Across 3 Controlled Seeds
# ============================================

import torch
import numpy as np
import re
import math
from tqdm import tqdm
from transformers import RobertaForMaskedLM, RobertaTokenizerFast

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# --------------------------------------------
# Load Tokenizer (same as training)
# --------------------------------------------
TOKENIZER_DIR = "/content/drive/MyDrive/MUSE_artifacts/babyroberta_tokenizer"

tokenizer = RobertaTokenizerFast.from_pretrained(TOKENIZER_DIR)

# --------------------------------------------
# Pseudo-Perplexity Function
# --------------------------------------------
def compute_pseudo_perplexity(text, tokenizer, model, max_length=512):
    model.eval()

    encodings = tokenizer(
        text,
        return_tensors="pt",
        truncation=True,
        max_length=max_length
    )

    input_ids = encodings["input_ids"].to(device)
    attention_mask = encodings["attention_mask"].to(device)

    seq_len = input_ids.size(1)
    log_likelihood = 0.0
    token_count = 0

    with torch.no_grad():
        for i in range(1, seq_len - 1):
            masked_input = input_ids.clone()
            masked_input[0, i] = tokenizer.mask_token_id

            outputs = model(masked_input, attention_mask=attention_mask)
            logits = outputs.logits

            log_probs = torch.log_softmax(logits[0, i], dim=-1)
            true_token = input_ids[0, i]

            log_likelihood += log_probs[true_token].item()
            token_count += 1

    if token_count == 0:
        return None

    ppl = math.exp(-log_likelihood / token_count)
    return ppl


# --------------------------------------------
# 30M Controlled Models
# --------------------------------------------
model_paths = [
    "/content/drive/MyDrive/MUSE_artifacts/babyroberta_30M_seed42_final",
    "/content/drive/MyDrive/MUSE_artifacts/babyroberta_30M_seed123_final",
    "/content/drive/MyDrive/MUSE_artifacts/babyroberta_30M_seed999_final",
]

# --------------------------------------------
# Evaluation Datasets
# --------------------------------------------
essay_files = {
    "Native_English": "/content/drive/MyDrive/SLA_Project/native_wikitext.txt",
    "Chinese_ICNALE": "/content/drive/MyDrive/SLA_Project/chinese_icnale_clean.txt",
    "Japanese_ICNALE": "/content/drive/MyDrive/SLA_Project/japanese_icnale_clean.txt",
    "Korean_ICNALE": "/content/drive/MyDrive/SLA_Project/kor_icnale_clean.txt",
    "Thai_ICNALE": "/content/drive/MyDrive/SLA_Project/tha_icnale_clean.txt",
    "Filipino_ICNALE": "/content/drive/MyDrive/SLA_Project/phl_icnale_clean.txt",
    "Urdu_ICNALE": "/content/drive/MyDrive/SLA_Project/urdu_icnale_clean.txt",

}

all_results = {}

# ============================================
# Loop Over Models
# ============================================
for model_path in model_paths:

    print("\n========================================")
    print("Loading model:", model_path)
    print("========================================")

    model = RobertaForMaskedLM.from_pretrained(model_path).to(device)
    model.eval()

    seed_results = {}

    for group, path in essay_files.items():

        print(f"\nEvaluating {group}...")

        ppl_scores = []

        # Chinese special handling
        if group == "Chinese_ICNALE":
            with open(path, "r", encoding="utf-8") as f:
                text = f.read()

            essays = re.split(r'(?=I\s)', text)
            essays = [e.strip() for e in essays if len(e.split()) > 50]

        else:
            with open(path, "r", encoding="utf-8") as f:
                essays = f.readlines()

        essays = essays[:200]

        for essay in tqdm(essays):
            essay = essay.strip()

            if len(essay.split()) < 5:
                continue

            ppl = compute_pseudo_perplexity(essay, tokenizer, model)

            if ppl is not None:
                ppl_scores.append(ppl)

        avg_ppl = np.mean(ppl_scores)
        seed_results[group] = avg_ppl

        print(f"{group} Average PPL: {avg_ppl:.3f}")

    all_results[model_path] = seed_results


# ============================================
# Final Averaging Across Seeds
# ============================================
print("\n\nFINAL AVERAGED RESULTS (Across Seeds)")
print("========================================")

for group in essay_files.keys():

    values = [all_results[m][group] for m in model_paths]

    mean_ppl = np.mean(values)
    std_ppl = np.std(values)

    print(f"{group}: Mean PPL = {mean_ppl:.3f} | Std = {std_ppl:.3f}")

In [ ]:
import shutil
import os

source_base = "/content/drive/MyDrive"
dest_base = "/content/drive/MyDrive/SLA_Project/WE/ICNALE_WE_2.6/WE_1_Classified_Unmerged"

folders = [
    "WE_ENS_XX1_N200",
    "WE_ENS_XX2_N088",
    "WE_ENS_XX3_N112"
]

for folder in folders:
    src = os.path.join(source_base, folder)
    dst = os.path.join(dest_base, folder)

    # Remove empty destination folder if it exists
    if os.path.exists(dst):
        shutil.rmtree(dst)

    shutil.move(src, dst)
    print(f"Moved {folder}")

print("Done.")

In [ ]:
import os

ens_path = "/content/drive/MyDrive/SLA_Project/WE/ICNALE_WE_2.6/WE_1_Classified_Unmerged/WE_ENS_XX1_N200"
print("ENS1 files:", len(os.listdir(ens_path)))

In [ ]:
import os

base = "/content/drive/MyDrive/SLA_Project/WE/ICNALE_WE_2.6/WE_1_Classified_Unmerged"

print("ENS1:", len(os.listdir(base + "/WE_ENS_XX1_N200")))
print("ENS2:", len(os.listdir(base + "/WE_ENS_XX2_N088")))
print("ENS3:", len(os.listdir(base + "/WE_ENS_XX3_N112")))

In [ ]:
import os

base = "/content/drive/MyDrive/SLA_Project/WE/ICNALE_WE_2.6/WE_1_Classified_Unmerged"

ens_folders = [
    "WE_ENS_XX1_N200",
    "WE_ENS_XX2_N088",
    "WE_ENS_XX3_N112"
]

all_texts = []

for folder in ens_folders:
    folder_path = os.path.join(base, folder)

    for file in os.listdir(folder_path):
        if file.endswith(".txt"):
            with open(os.path.join(folder_path, file), "r", encoding="utf-8", errors="ignore") as f:
                text = f.read().strip()
                all_texts.append(text)

print("Total ENS essays collected:", len(all_texts))

# Save combined file
output_path = "/content/drive/MyDrive/SLA_Project/ens_icnale_clean.txt"

with open(output_path, "w", encoding="utf-8") as out:
    for essay in all_texts:
        out.write(essay + "\n\n")

print("Saved to:", output_path)

In [ ]:
# ============================================
# Evaluate 30M Models Across 3 Controlled Seeds
# ============================================

import torch
import numpy as np
import re
import math
from tqdm import tqdm
from transformers import RobertaForMaskedLM, RobertaTokenizerFast

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# --------------------------------------------
# Load Tokenizer (same as training)
# --------------------------------------------
TOKENIZER_DIR = "/content/drive/MyDrive/MUSE_artifacts/babyroberta_tokenizer"

tokenizer = RobertaTokenizerFast.from_pretrained(TOKENIZER_DIR)

# --------------------------------------------
# Pseudo-Perplexity Function
# --------------------------------------------
def compute_pseudo_perplexity(text, tokenizer, model, max_length=512):
    model.eval()

    encodings = tokenizer(
        text,
        return_tensors="pt",
        truncation=True,
        max_length=max_length
    )

    input_ids = encodings["input_ids"].to(device)
    attention_mask = encodings["attention_mask"].to(device)

    seq_len = input_ids.size(1)
    log_likelihood = 0.0
    token_count = 0

    with torch.no_grad():
        for i in range(1, seq_len - 1):
            masked_input = input_ids.clone()
            masked_input[0, i] = tokenizer.mask_token_id

            outputs = model(masked_input, attention_mask=attention_mask)
            logits = outputs.logits

            log_probs = torch.log_softmax(logits[0, i], dim=-1)
            true_token = input_ids[0, i]

            log_likelihood += log_probs[true_token].item()
            token_count += 1

    if token_count == 0:
        return None

    ppl = math.exp(-log_likelihood / token_count)
    return ppl


# --------------------------------------------
# 30M Controlled Models
# --------------------------------------------
model_paths = [
    "/content/drive/MyDrive/MUSE_artifacts/babyroberta_30M_seed42_final",
    "/content/drive/MyDrive/MUSE_artifacts/babyroberta_30M_seed123_final",
    "/content/drive/MyDrive/MUSE_artifacts/babyroberta_30M_seed999_final",
]

# --------------------------------------------
# Evaluation Datasets
# --------------------------------------------
essay_files = {
    "Native_English_ICNALE_": "/content/drive/MyDrive/SLA_Project/ens_icnale_clean.txt",
    "Chinese_ICNALE": "/content/drive/MyDrive/SLA_Project/chinese_icnale_clean.txt",
    "Japanese_ICNALE": "/content/drive/MyDrive/SLA_Project/japanese_icnale_clean.txt",
    "Korean_ICNALE": "/content/drive/MyDrive/SLA_Project/kor_icnale_clean.txt",
    "Thai_ICNALE": "/content/drive/MyDrive/SLA_Project/tha_icnale_clean.txt",
    "Filipino_ICNALE": "/content/drive/MyDrive/SLA_Project/phl_icnale_clean.txt",
    "Urdu_ICNALE": "/content/drive/MyDrive/SLA_Project/urdu_icnale_clean.txt",
}

all_results = {}

# ============================================
# Loop Over Models
# ============================================
for model_path in model_paths:

    print("\n========================================")
    print("Loading model:", model_path)
    print("========================================")

    model = RobertaForMaskedLM.from_pretrained(model_path).to(device)
    model.eval()

    seed_results = {}

    for group, path in essay_files.items():

        print(f"\nEvaluating {group}...")

        ppl_scores = []

        # Chinese special handling
        if group == "Chinese_ICNALE":
            with open(path, "r", encoding="utf-8") as f:
                text = f.read()

            essays = re.split(r'(?=I\s)', text)
            essays = [e.strip() for e in essays if len(e.split()) > 50]

        else:
            with open(path, "r", encoding="utf-8") as f:
                essays = f.readlines()

        essays = essays[:200]

        for essay in tqdm(essays):
            essay = essay.strip()

            if len(essay.split()) < 5:
                continue

            ppl = compute_pseudo_perplexity(essay, tokenizer, model)

            if ppl is not None:
                ppl_scores.append(ppl)

        avg_ppl = np.mean(ppl_scores)
        seed_results[group] = avg_ppl

        print(f"{group} Average PPL: {avg_ppl:.3f}")

    all_results[model_path] = seed_results


# ============================================
# Final Averaging Across Seeds
# ============================================
print("\n\nFINAL AVERAGED RESULTS (Across Seeds)")
print("========================================")

for group in essay_files.keys():

    values = [all_results[m][group] for m in model_paths]

    mean_ppl = np.mean(values)
    std_ppl = np.std(values)

    print(f"{group}: Mean PPL = {mean_ppl:.3f} | Std = {std_ppl:.3f}")

In [ ]:
print(os.path.exists("/content/drive/MyDrive/MUSE_artifacts/babyroberta_tokenizer"))

In [ ]:
# ==============================
# Train 30M BabyRoBERTa (Seed 97)
# ==============================

import os
import random
import numpy as np
import torch
from google.colab import drive
from datasets import load_dataset
from transformers import (
    RobertaTokenizerFast,
    RobertaConfig,
    RobertaForMaskedLM,
    DataCollatorForLanguageModeling,
    TrainingArguments,
    Trainer,
    set_seed
)

# Mount Drive
drive.mount('/content/drive')

# ------------------------------
# Set Seed
# ------------------------------
SEED = 97

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
set_seed(SEED)

# ------------------------------
# Paths
# ------------------------------
CORPUS_PATH = "/content/drive/MyDrive/MUSE_artifacts/en_w2w_wiki_80M_v2_clean.txt"
TOKENIZER_DIR = "/content/drive/MyDrive/MUSE_artifacts/babyroberta_tokenizer"
OUTPUT_DIR = "/content/drive/MyDrive/MUSE_artifacts/babyroberta_30M_seed97"
FINAL_DIR = "/content/drive/MyDrive/MUSE_artifacts/babyroberta_30M_seed97_final"

# ------------------------------
# Load Tokenizer
# ------------------------------
tokenizer = RobertaTokenizerFast(
    tokenizer_file=f"{TOKENIZER_DIR}/tokenizer.json",
    max_len=512
)

# ------------------------------
# Create Model (Same Architecture)
# ------------------------------
config = RobertaConfig(
    vocab_size=32000,
    max_position_embeddings=514,
    num_attention_heads=8,
    num_hidden_layers=6,
    type_vocab_size=1,
    hidden_size=512,
    intermediate_size=2048,
)

model = RobertaForMaskedLM(config)

print("Model created with seed:", SEED)
print("Total parameters:", sum(p.numel() for p in model.parameters()) / 1e6, "Million")

# ------------------------------
# Load Dataset
# ------------------------------
dataset = load_dataset(
    "text",
    data_files={"train": CORPUS_PATH},
    streaming=False
)

def tokenize_function(examples):
    return tokenizer(
        examples["text"],
        truncation=True,
        max_length=512,
        return_special_tokens_mask=True,
    )

tokenized_dataset = dataset.map(
    tokenize_function,
    batched=True,
    remove_columns=["text"],
    num_proc=2
)

# ------------------------------
# Group into 512 token blocks
# ------------------------------
block_size = 512

def group_texts(examples):
    concatenated = {}
    for key in examples.keys():
        concatenated[key] = sum(examples[key], [])

    total_length = len(concatenated["input_ids"])
    total_length = (total_length // block_size) * block_size

    result = {}
    for key in concatenated.keys():
        result[key] = [
            concatenated[key][i : i + block_size]
            for i in range(0, total_length, block_size)
        ]

    return result

lm_dataset = tokenized_dataset.map(
    group_texts,
    batched=True,
    num_proc=2,
)

# ------------------------------
# MLM Data Collator
# ------------------------------
data_collator = DataCollatorForLanguageModeling(
    tokenizer=tokenizer,
    mlm=True,
    mlm_probability=0.15,
)

# ------------------------------
# Training Arguments
# ------------------------------
training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    num_train_epochs=5,
    per_device_train_batch_size=16,
    save_steps=2000,
    save_total_limit=2,
    prediction_loss_only=True,
    learning_rate=5e-4,
    weight_decay=0.01,
    warmup_steps=1000,
    logging_steps=500,
    fp16=True,
    report_to="none",
    seed=SEED,
)

# ------------------------------
# Trainer
# ------------------------------
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=lm_dataset["train"],
    data_collator=data_collator,
)

# ------------------------------
# Train (Fresh — No Resume)
# ------------------------------
trainer.train()

# ------------------------------
# Save Final Model
# ------------------------------
trainer.save_model(FINAL_DIR)
tokenizer.save_pretrained(FINAL_DIR)

print("Training complete.")
print("Model saved to:", FINAL_DIR)

In [ ]:
import os

path = "/content/drive/MyDrive/MUSE_artifacts/babyroberta_30M_seed97_final"

print("Exists:", os.path.exists(path))
print("Files:", os.listdir(path))

In [ ]:
# ==============================
# Train 30M BabyRoBERTa (Seed 420)
# ==============================

import os
import random
import numpy as np
import torch
from google.colab import drive
from datasets import load_dataset
from transformers import (
    RobertaTokenizerFast,
    RobertaConfig,
    RobertaForMaskedLM,
    DataCollatorForLanguageModeling,
    TrainingArguments,
    Trainer,
    set_seed
)

# Mount Drive
drive.mount('/content/drive')

# ------------------------------
# Set Seed
# ------------------------------
SEED = 420

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
set_seed(SEED)

# ------------------------------
# Paths
# ------------------------------
CORPUS_PATH = "/content/drive/MyDrive/MUSE_artifacts/en_w2w_wiki_80M_v2_clean.txt"
TOKENIZER_DIR = "/content/drive/MyDrive/MUSE_artifacts/babyroberta_tokenizer"
OUTPUT_DIR = "/content/drive/MyDrive/MUSE_artifacts/babyroberta_30M_seed420"
FINAL_DIR = "/content/drive/MyDrive/MUSE_artifacts/babyroberta_30M_seed420_final"

# ------------------------------
# Load Tokenizer
# ------------------------------
tokenizer = RobertaTokenizerFast(
    tokenizer_file=f"{TOKENIZER_DIR}/tokenizer.json",
    max_len=512
)

# ------------------------------
# Create Model (Same Architecture)
# ------------------------------
config = RobertaConfig(
    vocab_size=32000,
    max_position_embeddings=514,
    num_attention_heads=8,
    num_hidden_layers=6,
    type_vocab_size=1,
    hidden_size=512,
    intermediate_size=2048,
)

model = RobertaForMaskedLM(config)

print("Model created with seed:", SEED)
print("Total parameters:", sum(p.numel() for p in model.parameters()) / 1e6, "Million")

# ------------------------------
# Load Dataset
# ------------------------------
dataset = load_dataset(
    "text",
    data_files={"train": CORPUS_PATH},
    streaming=False
)

def tokenize_function(examples):
    return tokenizer(
        examples["text"],
        truncation=True,
        max_length=512,
        return_special_tokens_mask=True,
    )

tokenized_dataset = dataset.map(
    tokenize_function,
    batched=True,
    remove_columns=["text"],
    num_proc=2
)

# ------------------------------
# Group into 512 token blocks
# ------------------------------
block_size = 512

def group_texts(examples):
    concatenated = {}
    for key in examples.keys():
        concatenated[key] = sum(examples[key], [])

    total_length = len(concatenated["input_ids"])
    total_length = (total_length // block_size) * block_size

    result = {}
    for key in concatenated.keys():
        result[key] = [
            concatenated[key][i : i + block_size]
            for i in range(0, total_length, block_size)
        ]

    return result

lm_dataset = tokenized_dataset.map(
    group_texts,
    batched=True,
    num_proc=2,
)

# ------------------------------
# MLM Data Collator
# ------------------------------
data_collator = DataCollatorForLanguageModeling(
    tokenizer=tokenizer,
    mlm=True,
    mlm_probability=0.15,
)

# ------------------------------
# Training Arguments
# ------------------------------
training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    num_train_epochs=5,
    per_device_train_batch_size=16,
    save_steps=2000,
    save_total_limit=2,
    prediction_loss_only=True,
    learning_rate=5e-4,
    weight_decay=0.01,
    warmup_steps=1000,
    logging_steps=500,
    fp16=True,
    report_to="none",
    seed=SEED,
)

# ------------------------------
# Trainer
# ------------------------------
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=lm_dataset["train"],
    data_collator=data_collator,
)

# ------------------------------
# Train (Fresh — No Resume)
# ------------------------------
trainer.train()

# ------------------------------
# Save Final Model
# ------------------------------
trainer.save_model(FINAL_DIR)
tokenizer.save_pretrained(FINAL_DIR)

print("Training complete.")
print("Model saved to:", FINAL_DIR)

In [ ]:
import os

path = "/content/drive/MyDrive/MUSE_artifacts/babyroberta_30M_seed420_final"

print("Exists:", os.path.exists(path))
print("Files:", os.listdir(path))

In [ ]:
# ============================================
# Evaluate 30M Models Across 2 more (97, 420)Controlled Seeds
# ============================================

import torch
import numpy as np
import re
import math
from tqdm import tqdm
from transformers import RobertaForMaskedLM, RobertaTokenizerFast

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# --------------------------------------------
# Load Tokenizer (same as training)
# --------------------------------------------
TOKENIZER_DIR = "/content/drive/MyDrive/MUSE_artifacts/babyroberta_tokenizer"

tokenizer = RobertaTokenizerFast.from_pretrained(TOKENIZER_DIR)

# --------------------------------------------
# Pseudo-Perplexity Function
# --------------------------------------------
def compute_pseudo_perplexity(text, tokenizer, model, max_length=512):
    model.eval()

    encodings = tokenizer(
        text,
        return_tensors="pt",
        truncation=True,
        max_length=max_length
    )

    input_ids = encodings["input_ids"].to(device)
    attention_mask = encodings["attention_mask"].to(device)

    seq_len = input_ids.size(1)
    log_likelihood = 0.0
    token_count = 0

    with torch.no_grad():
        for i in range(1, seq_len - 1):
            masked_input = input_ids.clone()
            masked_input[0, i] = tokenizer.mask_token_id

            outputs = model(masked_input, attention_mask=attention_mask)
            logits = outputs.logits

            log_probs = torch.log_softmax(logits[0, i], dim=-1)
            true_token = input_ids[0, i]

            log_likelihood += log_probs[true_token].item()
            token_count += 1

    if token_count == 0:
        return None

    ppl = math.exp(-log_likelihood / token_count)
    return ppl


# --------------------------------------------
# 30M Controlled Models
# --------------------------------------------
model_paths = [
    "/content/drive/MyDrive/MUSE_artifacts/babyroberta_30M_seed97_final",
    "/content/drive/MyDrive/MUSE_artifacts/babyroberta_30M_seed420_final",
]

# --------------------------------------------
# Evaluation Datasets
# --------------------------------------------
essay_files = {
    "Native_English": "/content/drive/MyDrive/SLA_Project/native_wikitext.txt",
    "Chinese_ICNALE": "/content/drive/MyDrive/SLA_Project/chinese_icnale_clean.txt",
    "Japanese_ICNALE": "/content/drive/MyDrive/SLA_Project/japanese_icnale_clean.txt",
    "Korean_ICNALE": "/content/drive/MyDrive/SLA_Project/kor_icnale_clean.txt",
    "Thai_ICNALE": "/content/drive/MyDrive/SLA_Project/tha_icnale_clean.txt",
    "Filipino_ICNALE": "/content/drive/MyDrive/SLA_Project/phl_icnale_clean.txt",
    "Urdu_ICNALE": "/content/drive/MyDrive/SLA_Project/urdu_icnale_clean.txt",

}

all_results = {}

# ============================================
# Loop Over Models
# ============================================
for model_path in model_paths:

    print("\n========================================")
    print("Loading model:", model_path)
    print("========================================")

    model = RobertaForMaskedLM.from_pretrained(model_path).to(device)
    model.eval()

    seed_results = {}

    for group, path in essay_files.items():

        print(f"\nEvaluating {group}...")

        ppl_scores = []

        # Chinese special handling
        if group == "Chinese_ICNALE":
            with open(path, "r", encoding="utf-8") as f:
                text = f.read()

            essays = re.split(r'(?=I\s)', text)
            essays = [e.strip() for e in essays if len(e.split()) > 50]

        else:
            with open(path, "r", encoding="utf-8") as f:
                essays = f.readlines()

        essays = essays[:200]

        for essay in tqdm(essays):
            essay = essay.strip()

            if len(essay.split()) < 5:
                continue

            ppl = compute_pseudo_perplexity(essay, tokenizer, model)

            if ppl is not None:
                ppl_scores.append(ppl)

        avg_ppl = np.mean(ppl_scores)
        seed_results[group] = avg_ppl

        print(f"{group} Average PPL: {avg_ppl:.3f}")

    all_results[model_path] = seed_results


# ============================================
# Final Averaging Across Seeds
# ============================================
print("\n\nFINAL AVERAGED RESULTS (Across Seeds)")
print("========================================")

for group in essay_files.keys():

    values = [all_results[m][group] for m in model_paths]

    mean_ppl = np.mean(values)
    std_ppl = np.std(values)


In [ ]:
# Full corrected ONE-CELL code (RoBERTa-safe) — writes separate CSVs per model

import os
import torch
import pandas as pd
from transformers import AutoTokenizer, AutoModelForMaskedLM

# -------------------------------------------------
# CONFIG
# -------------------------------------------------
INPUT_FILE = "/content/drive/MyDrive/Mandarin_Question_MASK.txt"
OPTIONS_FILE = "/content/drive/MyDrive/Mandarin_answers.txt"

OUTPUT_DIR = "/content/drive/MyDrive/Mandarin_SLA_Project_CSEE"
os.makedirs(OUTPUT_DIR, exist_ok=True)

# ---- LIST ALL MODEL DIRECTORIES HERE ----
MODEL_DIRS = [
    "/content/drive/MyDrive/MUSE_artifacts/babyroberta_30M_seed42_final",
    "/content/drive/MyDrive/MUSE_artifacts/babyroberta_30M_seed97_final",
    "/content/drive/MyDrive/MUSE_artifacts/babyroberta_30M_seed123_final",
    "/content/drive/MyDrive/MUSE_artifacts/babyroberta_30M_seed420_final",
    "/content/drive/MyDrive/MUSE_artifacts/babyroberta_30M_seed999_final",
]

OPTIONS = [
    "on", "at", "like", "as", "with", "inside", "of", "among", "in",
    "by", "from", "during", "about", "near", "out", "round", "until",
    "along", "for", "against", "across", "to", "off", "upon", "towards",
    "under", "around", "behind", "besides", "within", "beside", "into",
    "between", "up", "over", "before", "above", "down", "after", "till",
    "toward", "without"
]

# -------------------------------------------------
# DEVICE
# -------------------------------------------------
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

# -------------------------------------------------
# LOAD SENTENCES (ONCE)
# -------------------------------------------------
sentences = []
with open(INPUT_FILE, "r", encoding="utf-8") as f:
    for line in f:
        line = line.strip()
        if not line:
            continue

        # Normalize common blank patterns to [MASK]
        line = line.replace("____", "[MASK]")
        line = line.replace(" ___ ", " [MASK] ")
        line = line.replace("___", "[MASK]")
        line = line.replace("__", "[MASK]")

        # If still no mask, insert near end
        if "[MASK]" not in line:
            parts = line.split()
            if len(parts) > 1:
                parts.insert(-1, "[MASK]")
                line = " ".join(parts)
            else:
                line = line + " [MASK]"

        sentences.append(line)

print("Loaded", len(sentences), "sentences.")

# -------------------------------------------------
# LOAD OPTION LINES (ONCE)
# -------------------------------------------------
with open(OPTIONS_FILE, "r", encoding="utf-8") as f:
    option_lines = [line.strip() for line in f if line.strip()]

if len(option_lines) != len(sentences):
    raise ValueError(
        f"OPTIONS_FILE line count ({len(option_lines)}) must match INPUT_FILE line count ({len(sentences)})."
    )

# -------------------------------------------------
# LOOP OVER MODELS
# -------------------------------------------------
for model_dir in MODEL_DIRS:
    model_name = os.path.basename(model_dir.rstrip("/"))
    print("\n========================================")
    print("Running model:", model_name)
    print("Model path:", model_dir)
    print("========================================")

    # Load tokenizer/model from SAME directory (keeps architecture + vocab consistent)
    tokenizer = AutoTokenizer.from_pretrained(model_dir, use_fast=True)
    model = AutoModelForMaskedLM.from_pretrained(model_dir)
    model.to(device)
    model.eval()

    # Precompute token IDs for OPTIONS (faster + safer)
    option_token_ids = {}
    for w in OPTIONS:
        tid = tokenizer.convert_tokens_to_ids(w)
        option_token_ids[w] = tid

    mask_token = tokenizer.mask_token
    mask_token_id = tokenizer.mask_token_id
    if mask_token is None or mask_token_id is None:
        raise ValueError(f"[ERROR] Tokenizer for {model_name} has no mask token. It must be an MLM tokenizer.")

    # -------------------------------------------------
    # MASK PREDICTION FUNCTION
    # -------------------------------------------------
    def predict_masked_word(sentence: str):
        # Ensure sentence has correct mask token for this tokenizer
        sent = sentence.replace("[MASK]", mask_token)

        inputs = tokenizer(sent, return_tensors="pt")
        inputs = {k: v.to(device) for k, v in inputs.items()}

        # Find first [MASK]
        mask_positions = (inputs["input_ids"] == mask_token_id).nonzero(as_tuple=False)
        if mask_positions.numel() == 0:
            return {w: 0.0 for w in OPTIONS}

        # mask_positions shape: [num_masks, 2] (batch_idx, token_idx)
        token_idx = int(mask_positions[0, 1].item())

        with torch.no_grad():
            logits = model(**inputs).logits  # [1, seq_len, vocab]

        mask_logits = logits[0, token_idx, :]  # [vocab]
        probs = torch.softmax(mask_logits, dim=-1)

        word_probs = {}
        for w in OPTIONS:
            wid = option_token_ids[w]
            if wid is None or wid < 0 or wid >= probs.size(-1):
                word_probs[w] = 0.0
            else:
                word_probs[w] = float(probs[wid].item())

        return word_probs

    # -------------------------------------------------
    # RAW PREDICTIONS
    # -------------------------------------------------
    rows = []
    for s in sentences:
        probs = predict_masked_word(s)
        row = {"Question": s}
        row.update(probs)
        rows.append(row)

    df_raw = pd.DataFrame(rows)
    raw_path = f"{OUTPUT_DIR}/{model_name}_raw.csv"
    df_raw.to_csv(raw_path, index=False, encoding="utf-8-sig")
    print("Raw predictions saved to:", raw_path)

    # -------------------------------------------------
    # NORMALIZED SCORES (only among allowed options per item)
    # -------------------------------------------------
    normalized_rows = []
    for i, allowed in enumerate(option_lines):
        allowed_list = [w.strip() for w in allowed.split(",") if w.strip()]
        sent = sentences[i]

        total = 0.0
        for w in allowed_list:
            if w in df_raw.columns:
                total += float(df_raw.loc[i, w])

        row = {"sentence": sent}
        for w in OPTIONS:
            if (w in allowed_list) and (total > 0):
                row[w] = float(df_raw.loc[i, w]) / total
            else:
                row[w] = 0.0

        normalized_rows.append(row)

    df_norm = pd.DataFrame(normalized_rows)
    norm_path = f"{OUTPUT_DIR}/{model_name}_normalized.csv"
    df_norm.to_csv(norm_path, index=False, encoding="utf-8-sig")
    print("Normalized scores saved to:", norm_path)

    # Free GPU memory
    del model
    torch.cuda.empty_cache()

print("\nALL MODELS FINISHED.")

In [ ]:
!wget https://dumps.wikimedia.org/simplewiki/latest/simplewiki-latest-pages-articles.xml.bz2
!bunzip2 simplewiki-latest-pages-articles.xml.bz2

In [ ]:
import re

INPUT_XML = "simplewiki-latest-pages-articles.xml"
OUTPUT_TXT = "/content/simplewiki_clean.txt"

with open(INPUT_XML, "r", encoding="utf-8", errors="ignore") as f:
    text = f.read()

# remove xml tags
text = re.sub(r"<.*?>", " ", text)

# clean spaces
lines = [l.strip() for l in text.split("\n") if len(l.split()) > 5]

with open(OUTPUT_TXT, "w") as f:
    for l in lines:
        f.write(l + "\n")

print("Saved clean SimpleWiki:", OUTPUT_TXT)

In [ ]:
import random

MAIN_PATH = "/content/drive/MyDrive/MUSE_artifacts/en_w2w_wiki_80M_v2_clean.txt"
OUTPUT_MAIN_25M = "/content/main_25M.txt"

target_words = 25_000_000
current_words = 0
selected_lines = []

with open(MAIN_PATH, "r", encoding="utf-8") as f:
    lines = f.readlines()

random.shuffle(lines)

for line in lines:
    words = line.strip().split()
    if len(words) < 5:
        continue

    selected_lines.append(line)
    current_words += len(words)

    if current_words >= target_words:
        break

with open(OUTPUT_MAIN_25M, "w") as f:
    f.writelines(selected_lines)

print("Saved 25M dataset")
print("Total words:", current_words)

In [ ]:
EN_PATH = "/content/simplewiki_clean.txt"
OUTPUT_EN_5M = "/content/english_5M.txt"

target_words = 5_000_000
current_words = 0
selected_lines = []

with open(EN_PATH, "r", encoding="utf-8") as f:
    lines = f.readlines()

random.shuffle(lines)

for line in lines:
    words = line.strip().split()
    if len(words) < 5:
        continue

    selected_lines.append(line)
    current_words += len(words)

    if current_words >= target_words:
        break

with open(OUTPUT_EN_5M, "w") as f:
    f.writelines(selected_lines)

print("Saved 5M English dataset")
print("Total words:", current_words)

In [ ]:
FINAL_DATASET = "/content/final_30M_mixed.txt"

with open(FINAL_DATASET, "w") as out:
    for path in [OUTPUT_MAIN_25M, OUTPUT_EN_5M]:
        with open(path, "r") as f:
            out.write(f.read())

print("Final dataset created:", FINAL_DATASET)

In [ ]:
with open(FINAL_DATASET, "r") as f:
    lines = f.readlines()

random.shuffle(lines)

with open(FINAL_DATASET, "w") as f:
    f.writelines(lines)

print("Dataset shuffled")

In [ ]:
# ==============================
# Train 5 BabyRoBERTa Models (Different Seeds)
# ==============================

import os
import random
import numpy as np
import torch
from google.colab import drive
from datasets import load_dataset
from transformers import (
    RobertaTokenizerFast,
    RobertaConfig,
    RobertaForMaskedLM,
    DataCollatorForLanguageModeling,
    TrainingArguments,
    Trainer,
    set_seed
)

# ------------------------------
# Mount Drive
# ------------------------------
drive.mount('/content/drive')

# ------------------------------
# Seeds to Train
# ------------------------------
SEEDS = [42, 97, 123, 420, 999]

# ------------------------------
# Paths
# ------------------------------
CORPUS_PATH = "/content/final_30M_mixed.txt"  # your mixed dataset
TOKENIZER_DIR = "/content/drive/MyDrive/MUSE_artifacts/babyroberta_tokenizer"
BASE_OUTPUT = "/content/drive/MyDrive/MUSE_artifacts"

# ------------------------------
# Load Tokenizer ONCE
# ------------------------------
tokenizer = RobertaTokenizerFast(
    tokenizer_file=f"{TOKENIZER_DIR}/tokenizer.json",
    max_len=512
)

# ------------------------------
# Load Dataset ONCE (FAST)
# ------------------------------
dataset = load_dataset(
    "text",
    data_files={"train": CORPUS_PATH},
    streaming=False
)

def tokenize_function(examples):
    return tokenizer(
        examples["text"],
        truncation=True,
        max_length=512,
        return_special_tokens_mask=True,
    )

tokenized_dataset = dataset.map(
    tokenize_function,
    batched=True,
    remove_columns=["text"],
    num_proc=2
)

# ------------------------------
# Group into 512 blocks
# ------------------------------
block_size = 512

def group_texts(examples):
    concatenated = {}
    for key in examples.keys():
        concatenated[key] = sum(examples[key], [])

    total_length = len(concatenated["input_ids"])
    total_length = (total_length // block_size) * block_size

    result = {}
    for key in concatenated.keys():
        result[key] = [
            concatenated[key][i : i + block_size]
            for i in range(0, total_length, block_size)
        ]

    return result

lm_dataset = tokenized_dataset.map(
    group_texts,
    batched=True,
    num_proc=2,
)

# ------------------------------
# MLM Data Collator
# ------------------------------
data_collator = DataCollatorForLanguageModeling(
    tokenizer=tokenizer,
    mlm=True,
    mlm_probability=0.15,
)

# ==============================
# LOOP OVER SEEDS
# ==============================
for SEED in SEEDS:

    print("\n==============================")
    print(f"Training Seed: {SEED}")
    print("==============================")

    # ------------------------------
    # Set Seed
    # ------------------------------
    random.seed(SEED)
    np.random.seed(SEED)
    torch.manual_seed(SEED)
    torch.cuda.manual_seed_all(SEED)
    set_seed(SEED)

    # ------------------------------
    # Create Model (fresh each time)
    # ------------------------------
    config = RobertaConfig(
        vocab_size=32000,
        max_position_embeddings=514,
        num_attention_heads=8,
        num_hidden_layers=6,
        type_vocab_size=1,
        hidden_size=512,
        intermediate_size=2048,
    )

    model = RobertaForMaskedLM(config)

    print("Model created with seed:", SEED)

    # ------------------------------
    # Output paths
    # ------------------------------
    OUTPUT_DIR = f"{BASE_OUTPUT}/babyroberta_30M_mixed_seed{SEED}"
    FINAL_DIR = f"{BASE_OUTPUT}/babyroberta_30M_mixed_seed{SEED}_final"

    # ------------------------------
    # Training Arguments
    # ------------------------------
    training_args = TrainingArguments(
        output_dir=OUTPUT_DIR,
        num_train_epochs=5,
        per_device_train_batch_size=16,
        save_steps=2000,
        save_total_limit=2,
        prediction_loss_only=True,
        learning_rate=5e-4,
        weight_decay=0.01,
        warmup_steps=1000,
        logging_steps=500,
        fp16=True,
        report_to="none",
        seed=SEED,
    )

    # ------------------------------
    # Trainer
    # ------------------------------
    trainer = Trainer(
        model=model,
        args=training_args,
        train_dataset=lm_dataset["train"],
        data_collator=data_collator,
    )

    # ------------------------------
    # Train
    # ------------------------------
    trainer.train()

    # ------------------------------
    # Save model safely
    # ------------------------------
    trainer.save_model(FINAL_DIR)
    tokenizer.save_pretrained(FINAL_DIR)

    print("Saved model:", FINAL_DIR)

print("\n ALL MODELS TRAINED SUCCESSFULLY ")

In [ ]:
import os

BASE = "/content/drive/MyDrive/MUSE_artifacts"

for seed in [42,97,123,420,999]:
    path = f"{BASE}/babyroberta_30M_mixed_seed{seed}_final"
    print(f"\nSeed {seed}:")
    print("Exists:", os.path.exists(path))
    print("Files:", os.listdir(path))

In [ ]:
# ============================================
# Evaluate 25M chinese transalted to english + 5M englishModels Across 5 Controlled Seeds
# ============================================

import torch
import numpy as np
import re
import math
from tqdm import tqdm
from transformers import RobertaForMaskedLM, RobertaTokenizerFast

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# --------------------------------------------
# Load Tokenizer (same as training)
# --------------------------------------------
TOKENIZER_DIR = "/content/drive/MyDrive/MUSE_artifacts/babyroberta_tokenizer"

tokenizer = RobertaTokenizerFast.from_pretrained(TOKENIZER_DIR)

# --------------------------------------------
# Pseudo-Perplexity Function
# --------------------------------------------
def compute_pseudo_perplexity(text, tokenizer, model, max_length=512):
    model.eval()

    encodings = tokenizer(
        text,
        return_tensors="pt",
        truncation=True,
        max_length=max_length
    )

    input_ids = encodings["input_ids"].to(device)
    attention_mask = encodings["attention_mask"].to(device)

    seq_len = input_ids.size(1)
    log_likelihood = 0.0
    token_count = 0

    with torch.no_grad():
        for i in range(1, seq_len - 1):
            masked_input = input_ids.clone()
            masked_input[0, i] = tokenizer.mask_token_id

            outputs = model(masked_input, attention_mask=attention_mask)
            logits = outputs.logits

            log_probs = torch.log_softmax(logits[0, i], dim=-1)
            true_token = input_ids[0, i]

            log_likelihood += log_probs[true_token].item()
            token_count += 1

    if token_count == 0:
        return None

    ppl = math.exp(-log_likelihood / token_count)
    return ppl


# --------------------------------------------
# 5 MODELS (UPDATED)
# --------------------------------------------
model_paths = [
    "/content/drive/MyDrive/MUSE_artifacts/babyroberta_30M_mixed_seed42_final",
    "/content/drive/MyDrive/MUSE_artifacts/babyroberta_30M_mixed_seed97_final",
    "/content/drive/MyDrive/MUSE_artifacts/babyroberta_30M_mixed_seed123_final",
    "/content/drive/MyDrive/MUSE_artifacts/babyroberta_30M_mixed_seed420_final",
    "/content/drive/MyDrive/MUSE_artifacts/babyroberta_30M_mixed_seed999_final",
]

# --------------------------------------------
# Evaluation Datasets
# --------------------------------------------
essay_files = {
    "Native_English": "/content/drive/MyDrive/SLA_Project/native_wikitext.txt",
    "Native_English_ICNALE": "/content/drive/MyDrive/SLA_Project/ens_icnale_clean.txt",
    "Chinese_ICNALE": "/content/drive/MyDrive/SLA_Project/chinese_icnale_clean.txt",
    "Japanese_ICNALE": "/content/drive/MyDrive/SLA_Project/japanese_icnale_clean.txt",
    "Korean_ICNALE": "/content/drive/MyDrive/SLA_Project/kor_icnale_clean.txt",
    "Thai_ICNALE": "/content/drive/MyDrive/SLA_Project/tha_icnale_clean.txt",
    "Filipino_ICNALE": "/content/drive/MyDrive/SLA_Project/phl_icnale_clean.txt",
    "Urdu_ICNALE": "/content/drive/MyDrive/SLA_Project/urdu_icnale_clean.txt",
}

all_results = {}

# ============================================
# Loop Over Models
# ============================================
for model_path in model_paths:

    print("\n========================================")
    print("Loading model:", model_path)
    print("========================================")

    model = RobertaForMaskedLM.from_pretrained(model_path).to(device)
    model.eval()

    seed_results = {}

    for group, path in essay_files.items():

        print(f"\nEvaluating {group}...")

        ppl_scores = []

        # Chinese special handling
        if group == "Chinese_ICNALE":
            with open(path, "r", encoding="utf-8") as f:
                text = f.read()

            essays = re.split(r'(?=I\s)', text)
            essays = [e.strip() for e in essays if len(e.split()) > 50]

        else:
            with open(path, "r", encoding="utf-8") as f:
                essays = f.readlines()

        essays = essays[:200]

        for essay in tqdm(essays):
            essay = essay.strip()

            if len(essay.split()) < 5:
                continue

            ppl = compute_pseudo_perplexity(essay, tokenizer, model)

            if ppl is not None:
                ppl_scores.append(ppl)

        avg_ppl = np.mean(ppl_scores)
        seed_results[group] = avg_ppl

        print(f"{group} Average PPL: {avg_ppl:.3f}")

    all_results[model_path] = seed_results


# ============================================
# Final Averaging Across Seeds
# ============================================
print("\n\nFINAL AVERAGED RESULTS (Across 5 Seeds)")
print("========================================")

for group in essay_files.keys():

    values = [all_results[m][group] for m in model_paths]

    mean_ppl = np.mean(values)
    std_ppl = np.std(values)

    print(f"{group} → Mean PPL: {mean_ppl:.3f} | Std: {std_ppl:.3f}")

In [ ]:
# Full corrected ONE-CELL code (RoBERTa-safe) — writes separate CSVs per model

import os
import torch
import pandas as pd
from transformers import AutoTokenizer, AutoModelForMaskedLM

# -------------------------------------------------
# CONFIG
# -------------------------------------------------
INPUT_FILE = "/content/drive/MyDrive/Mandarin_Question_MASK.txt"
OPTIONS_FILE = "/content/drive/MyDrive/Mandarin_answers.txt"

OUTPUT_DIR = "/content/drive/MyDrive/Mandarin_SLA_Project_CSEE"
os.makedirs(OUTPUT_DIR, exist_ok=True)

# ---- UPDATED: LATEST MIXED MODELS ----
MODEL_DIRS = [
    "/content/drive/MyDrive/MUSE_artifacts/babyroberta_30M_mixed_seed42_final",
    "/content/drive/MyDrive/MUSE_artifacts/babyroberta_30M_mixed_seed97_final",
    "/content/drive/MyDrive/MUSE_artifacts/babyroberta_30M_mixed_seed123_final",
    "/content/drive/MyDrive/MUSE_artifacts/babyroberta_30M_mixed_seed420_final",
    "/content/drive/MyDrive/MUSE_artifacts/babyroberta_30M_mixed_seed999_final",
]

OPTIONS = [
    "on", "at", "like", "as", "with", "inside", "of", "among", "in",
    "by", "from", "during", "about", "near", "out", "round", "until",
    "along", "for", "against", "across", "to", "off", "upon", "towards",
    "under", "around", "behind", "besides", "within", "beside", "into",
    "between", "up", "over", "before", "above", "down", "after", "till",
    "toward", "without"
]

# -------------------------------------------------
# DEVICE
# -------------------------------------------------
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

# -------------------------------------------------
# LOAD SENTENCES (ONCE)
# -------------------------------------------------
sentences = []
with open(INPUT_FILE, "r", encoding="utf-8") as f:
    for line in f:
        line = line.strip()
        if not line:
            continue

        line = line.replace("____", "[MASK]")
        line = line.replace(" ___ ", " [MASK] ")
        line = line.replace("___", "[MASK]")
        line = line.replace("__", "[MASK]")

        if "[MASK]" not in line:
            parts = line.split()
            if len(parts) > 1:
                parts.insert(-1, "[MASK]")
                line = " ".join(parts)
            else:
                line = line + " [MASK]"

        sentences.append(line)

print("Loaded", len(sentences), "sentences.")

# -------------------------------------------------
# LOAD OPTION LINES (ONCE)
# -------------------------------------------------
with open(OPTIONS_FILE, "r", encoding="utf-8") as f:
    option_lines = [line.strip() for line in f if line.strip()]

if len(option_lines) != len(sentences):
    raise ValueError(
        f"OPTIONS_FILE line count ({len(option_lines)}) must match INPUT_FILE line count ({len(sentences)})."
    )

# -------------------------------------------------
# LOOP OVER MODELS
# -------------------------------------------------
for model_dir in MODEL_DIRS:
    model_name = os.path.basename(model_dir.rstrip("/"))
    print("\n========================================")
    print("Running model:", model_name)
    print("========================================")

    tokenizer = AutoTokenizer.from_pretrained(model_dir, use_fast=True)
    model = AutoModelForMaskedLM.from_pretrained(model_dir)
    model.to(device)
    model.eval()

    option_token_ids = {w: tokenizer.convert_tokens_to_ids(w) for w in OPTIONS}

    mask_token = tokenizer.mask_token
    mask_token_id = tokenizer.mask_token_id

    def predict_masked_word(sentence):
        sent = sentence.replace("[MASK]", mask_token)

        inputs = tokenizer(sent, return_tensors="pt")
        inputs = {k: v.to(device) for k, v in inputs.items()}

        mask_positions = (inputs["input_ids"] == mask_token_id).nonzero(as_tuple=False)
        if mask_positions.numel() == 0:
            return {w: 0.0 for w in OPTIONS}

        token_idx = int(mask_positions[0, 1].item())

        with torch.no_grad():
            logits = model(**inputs).logits

        probs = torch.softmax(logits[0, token_idx], dim=-1)

        return {
            w: float(probs[option_token_ids[w]].item())
            if option_token_ids[w] is not None else 0.0
            for w in OPTIONS
        }

    # -------------------------------------------------
    # RAW
    # -------------------------------------------------
    rows = []
    for s in sentences:
        row = {"Question": s}
        row.update(predict_masked_word(s))
        rows.append(row)

    df_raw = pd.DataFrame(rows)
    raw_path = f"{OUTPUT_DIR}/{model_name}_raw.csv"
    df_raw.to_csv(raw_path, index=False)
    print("Saved:", raw_path)

    # -------------------------------------------------
    # NORMALIZED
    # -------------------------------------------------
    normalized_rows = []
    for i, allowed in enumerate(option_lines):
        allowed_list = [w.strip() for w in allowed.split(",") if w.strip()]

        total = sum(df_raw.loc[i, w] for w in allowed_list if w in df_raw.columns)

        row = {"sentence": sentences[i]}
        for w in OPTIONS:
            row[w] = df_raw.loc[i, w] / total if (w in allowed_list and total > 0) else 0.0

        normalized_rows.append(row)

    df_norm = pd.DataFrame(normalized_rows)
    norm_path = f"{OUTPUT_DIR}/{model_name}_normalized.csv"
    df_norm.to_csv(norm_path, index=False)
    print("Saved:", norm_path)

    del model
    torch.cuda.empty_cache()

print("\nALL MODELS FINISHED.")

In [ ]:
# Full corrected ONE-CELL code (BERT-base-uncased) — SAME STRUCTURE

import os
import torch
import pandas as pd
from transformers import AutoTokenizer, AutoModelForMaskedLM

# -------------------------------------------------
# CONFIG
# -------------------------------------------------
INPUT_FILE = "/content/drive/MyDrive/Mandarin_Question_MASK.txt"
OPTIONS_FILE = "/content/drive/MyDrive/Mandarin_answers.txt"

OUTPUT_DIR = "/content/drive/MyDrive/Mandarin_SLA_Project_CSEE"
os.makedirs(OUTPUT_DIR, exist_ok=True)

# ---- ONLY ONE MODEL (Untuned BERT) ----
MODEL_DIRS = [
    "bert-base-uncased"
]

OPTIONS = [
    "on", "at", "like", "as", "with", "inside", "of", "among", "in",
    "by", "from", "during", "about", "near", "out", "round", "until",
    "along", "for", "against", "across", "to", "off", "upon", "towards",
    "under", "around", "behind", "besides", "within", "beside", "into",
    "between", "up", "over", "before", "above", "down", "after", "till",
    "toward", "without"
]

# -------------------------------------------------
# DEVICE
# -------------------------------------------------
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

# -------------------------------------------------
# LOAD SENTENCES (ONCE)
# -------------------------------------------------
sentences = []
with open(INPUT_FILE, "r", encoding="utf-8") as f:
    for line in f:
        line = line.strip()
        if not line:
            continue

        line = line.replace("____", "[MASK]")
        line = line.replace(" ___ ", " [MASK] ")
        line = line.replace("___", "[MASK]")
        line = line.replace("__", "[MASK]")

        if "[MASK]" not in line:
            parts = line.split()
            if len(parts) > 1:
                parts.insert(-1, "[MASK]")
                line = " ".join(parts)
            else:
                line = line + " [MASK]"

        sentences.append(line)

print("Loaded", len(sentences), "sentences.")

# -------------------------------------------------
# LOAD OPTION LINES (ONCE)
# -------------------------------------------------
with open(OPTIONS_FILE, "r", encoding="utf-8") as f:
    option_lines = [line.strip() for line in f if line.strip()]

if len(option_lines) != len(sentences):
    raise ValueError(
        f"OPTIONS_FILE line count ({len(option_lines)}) must match INPUT_FILE line count ({len(sentences)})."
    )

# -------------------------------------------------
# LOOP OVER MODELS
# -------------------------------------------------
for model_dir in MODEL_DIRS:
    model_name = "Base_BERT"
    print("\n========================================")
    print("Running model:", model_name)
    print("========================================")

    tokenizer = AutoTokenizer.from_pretrained(model_dir, use_fast=True)
    model = AutoModelForMaskedLM.from_pretrained(model_dir)
    model.to(device)
    model.eval()

    option_token_ids = {w: tokenizer.convert_tokens_to_ids(w) for w in OPTIONS}

    mask_token = tokenizer.mask_token
    mask_token_id = tokenizer.mask_token_id

    def predict_masked_word(sentence):
        sent = sentence.replace("[MASK]", mask_token)

        inputs = tokenizer(sent, return_tensors="pt")
        inputs = {k: v.to(device) for k, v in inputs.items()}

        mask_positions = (inputs["input_ids"] == mask_token_id).nonzero(as_tuple=False)
        if mask_positions.numel() == 0:
            return {w: 0.0 for w in OPTIONS}

        token_idx = int(mask_positions[0, 1].item())

        with torch.no_grad():
            logits = model(**inputs).logits

        probs = torch.softmax(logits[0, token_idx], dim=-1)

        return {
            w: float(probs[option_token_ids[w]].item())
            if option_token_ids[w] is not None else 0.0
            for w in OPTIONS
        }

    # -------------------------------------------------
    # RAW
    # -------------------------------------------------
    rows = []
    for s in sentences:
        row = {"Question": s}
        row.update(predict_masked_word(s))
        rows.append(row)

    df_raw = pd.DataFrame(rows)
    raw_path = f"{OUTPUT_DIR}/{model_name}_raw.csv"
    df_raw.to_csv(raw_path, index=False)
    print("Saved:", raw_path)

    # -------------------------------------------------
    # NORMALIZED
    # -------------------------------------------------
    normalized_rows = []
    for i, allowed in enumerate(option_lines):
        allowed_list = [w.strip() for w in allowed.split(",") if w.strip()]

        total = sum(df_raw.loc[i, w] for w in allowed_list if w in df_raw.columns)

        row = {"sentence": sentences[i]}
        for w in OPTIONS:
            row[w] = df_raw.loc[i, w] / total if (w in allowed_list and total > 0) else 0.0

        normalized_rows.append(row)

    df_norm = pd.DataFrame(normalized_rows)
    norm_path = f"{OUTPUT_DIR}/{model_name}_normalized.csv"
    df_norm.to_csv(norm_path, index=False)
    print("Saved:", norm_path)

    del model
    torch.cuda.empty_cache()

print("\nALL MODELS FINISHED.")

In [ ]:
CORPUS_PATH = "/content/drive/MyDrive/MUSE_artifacts/en_w2w_wiki_80M_v2_clean.txt"

PREPOSITIONS = {"in", "on", "at", "to", "for", "with", "of", "by", "from"}

In [ ]:
def read_data(file_path, max_lines=200000):
    with open(file_path, "r") as f:
        for i, line in enumerate(f):
            if i >= max_lines:
                break
            tokens = line.strip().split()
            if len(tokens) > 4:
                yield tokens

In [ ]:
def extract_ngram_features(tokens, i):
    features = {}

    # 2-gram (bigram)
    if i > 0 and i < len(tokens)-1:
        features["2gram"] = tokens[i-1] + "_" + tokens[i+1]

    # 3-gram (trigram)
    if i > 1 and i < len(tokens)-1:
        features["3gram"] = tokens[i-2] + "_" + tokens[i-1] + "_" + tokens[i+1]

    # 4-gram
    if i > 2 and i < len(tokens)-1:
        features["4gram"] = (
            tokens[i-3] + "_" + tokens[i-2] + "_" + tokens[i-1] + "_" + tokens[i+1]
        )

    return features

In [ ]:
def build_dataset(file_path, max_lines=200000):
    X = []
    y = []

    for tokens in read_data(file_path, max_lines):
        for i, word in enumerate(tokens):
            if word in PREPOSITIONS:
                features = extract_ngram_features(tokens, i)

                # skip if no features
                if len(features) > 0:
                    X.append(features)
                    y.append(word)

    return X, y

In [ ]:
X, y = build_dataset(CORPUS_PATH, max_lines=200000)

print("Total samples:", len(X))
print("Example:", X[0], y[0])

In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.feature_extraction import DictVectorizer
from sklearn.naive_bayes import MultinomialNB

model = Pipeline([
    ("vectorizer", DictVectorizer()),
    ("nb", MultinomialNB())
])

model.fit(X_train, y_train)

In [ ]:
accuracy = model.score(X_test, y_test)
print("Naive Bayes Accuracy (2-4 grams):", accuracy)

In [ ]:
sample = {
    "2gram": "go_school",
    "3gram": "I_go_school"
}

print("Prediction:", model.predict([sample])[0])

In [ ]:
from collections import Counter
print(Counter(y))

In [ ]:
baseline = y_test.count("of") / len(y_test)
print("Always predict 'of' accuracy:", baseline)

In [ ]:
import random

ENGLISH_PATH = "/content/simplewiki_clean.txt"
OUTPUT_PATH = "/content/english_5M.txt"

with open(ENGLISH_PATH, "r", encoding="utf-8") as f:
    lines = f.readlines()

random.shuffle(lines)

selected_lines = []
word_count = 0

for line in lines:
    words = line.split()

    if len(words) < 5:
        continue  # skip very short lines

    selected_lines.append(line.strip())
    word_count += len(words)

    if word_count >= 5_000_000:
        break

# Save
with open(OUTPUT_PATH, "w", encoding="utf-8") as f:
    for line in selected_lines:
        f.write(line + "\n")

print("Saved:", OUTPUT_PATH)
print("Total words:", word_count)
print("Total lines:", len(selected_lines))

In [ ]:
FINAL_PATH = "/content/final_30M_plus_5M.txt"

with open("/content/drive/MyDrive/MUSE_artifacts/en_w2w_wiki_80M_v2_clean.txt") as f:
    chinese = f.readlines()

with open("/content/english_5M.txt") as f:
    english = f.readlines()

final_data = chinese + english
random.shuffle(final_data)

with open(FINAL_PATH, "w") as f:
    for line in final_data:
        f.write(line)

print("Final dataset ready:", FINAL_PATH)

In [ ]:
FILE_PATH = "/content/final_30M_plus_5M.txt"

total_words = 0
total_lines = 0

with open(FILE_PATH, "r", encoding="utf-8") as f:
    for line in f:
        words = line.split()
        total_words += len(words)
        total_lines += 1

print("Total lines:", total_lines)
print("Total words:", total_words)

In [ ]:
import random

FILE_PATH = "/content/final_30M_plus_5M.txt"

with open(FILE_PATH, "r", encoding="utf-8") as f:
    lines = f.readlines()

random.shuffle(lines)

with open(FILE_PATH, "w", encoding="utf-8") as f:
    for line in lines:
        f.write(line)

print("Shuffled again successfully")

In [ ]:
# ==============================
# Train 5 BabyRoBERTa Models (Different Seeds)
# ==============================

import os
import random
import numpy as np
import torch
from google.colab import drive
from datasets import load_dataset
from transformers import (
    RobertaTokenizerFast,
    RobertaConfig,
    RobertaForMaskedLM,
    DataCollatorForLanguageModeling,
    TrainingArguments,
    Trainer,
    set_seed
)

# ------------------------------
# Mount Drive
# ------------------------------
drive.mount('/content/drive')

# ------------------------------
# Seeds to Train
# ------------------------------
SEEDS = [42, 97, 123, 420, 999]

# ------------------------------
# Paths
# ------------------------------
CORPUS_PATH = "/content/final_30M_plus_5M.txt"   # new created dataset with mixed
TOKENIZER_DIR = "/content/drive/MyDrive/MUSE_artifacts/babyroberta_tokenizer"
BASE_OUTPUT = "/content/drive/MyDrive/MUSE_artifacts"

# ------------------------------
# Load Tokenizer ONCE
# ------------------------------
tokenizer = RobertaTokenizerFast(
    tokenizer_file=f"{TOKENIZER_DIR}/tokenizer.json",
    max_len=512
)

# ------------------------------
# Load Dataset ONCE (FAST)
# ------------------------------
dataset = load_dataset(
    "text",
    data_files={"train": CORPUS_PATH},
    streaming=False
)

def tokenize_function(examples):
    return tokenizer(
        examples["text"],
        truncation=True,
        max_length=512,
        return_special_tokens_mask=True,
    )

tokenized_dataset = dataset.map(
    tokenize_function,
    batched=True,
    remove_columns=["text"],
    num_proc=2
)

# ------------------------------
# Group into 512 blocks
# ------------------------------
block_size = 512

def group_texts(examples):
    concatenated = {}
    for key in examples.keys():
        concatenated[key] = sum(examples[key], [])

    total_length = len(concatenated["input_ids"])
    total_length = (total_length // block_size) * block_size

    result = {}
    for key in concatenated.keys():
        result[key] = [
            concatenated[key][i : i + block_size]
            for i in range(0, total_length, block_size)
        ]

    return result

lm_dataset = tokenized_dataset.map(
    group_texts,
    batched=True,
    num_proc=2,
)

# ------------------------------
# MLM Data Collator
# ------------------------------
data_collator = DataCollatorForLanguageModeling(
    tokenizer=tokenizer,
    mlm=True,
    mlm_probability=0.15,
)

# ==============================
# LOOP OVER SEEDS
# ==============================
for SEED in SEEDS:

    print("\n==============================")
    print(f"Training Seed: {SEED}")
    print("==============================")

    # ------------------------------
    # Set Seed
    # ------------------------------
    random.seed(SEED)
    np.random.seed(SEED)
    torch.manual_seed(SEED)
    torch.cuda.manual_seed_all(SEED)
    set_seed(SEED)

    # ------------------------------
    # Create Model (fresh each time)
    # ------------------------------
    config = RobertaConfig(
        vocab_size=32000,
        max_position_embeddings=514,
        num_attention_heads=8,
        num_hidden_layers=6,
        type_vocab_size=1,
        hidden_size=512,
        intermediate_size=2048,
    )

    model = RobertaForMaskedLM(config)

    print("Model created with seed:", SEED)

    # ------------------------------
    # Output paths
    # ------------------------------
    OUTPUT_DIR = f"{BASE_OUTPUT}/babyroberta_30M_plus5M_seed{SEED}"
    FINAL_DIR = f"{BASE_OUTPUT}/babyroberta_30M_plus5M_seed{SEED}_final"

    # ------------------------------
    # Training Arguments
    # ------------------------------
    training_args = TrainingArguments(
        output_dir=OUTPUT_DIR,
        num_train_epochs=5,
        per_device_train_batch_size=16,
        save_steps=2000,
        save_total_limit=2,
        prediction_loss_only=True,
        learning_rate=5e-4,
        weight_decay=0.01,
        warmup_steps=1000,
        logging_steps=500,
        fp16=True,
        report_to="none",
        seed=SEED,
    )

    # ------------------------------
    # Trainer
    # ------------------------------
    trainer = Trainer(
        model=model,
        args=training_args,
        train_dataset=lm_dataset["train"],
        data_collator=data_collator,
    )

    # ------------------------------
    # Train
    # ------------------------------
    trainer.train()

    # ------------------------------
    # Save model safely
    # ------------------------------
    trainer.save_model(FINAL_DIR)
    tokenizer.save_pretrained(FINAL_DIR)

    print("Saved model:", FINAL_DIR)

print("\n ALL MODELS TRAINED SUCCESSFULLY ")

In [ ]:
import os

file_path = "/content/drive/MyDrive/Mandarin_Question_MASK.txt"

print(os.path.exists(file_path))

In [ ]:
!pip uninstall torch -y
!pip install torch

In [ ]:
# Full corrected ONE-CELL code (RoBERTa-safe) — writes separate CSVs per model

import os
import torch
import pandas as pd
from transformers import AutoTokenizer, AutoModelForMaskedLM

# -------------------------------------------------
# CONFIG
# -------------------------------------------------
INPUT_FILE = "/content/drive/MyDrive/Mandarin_Question_MASK.txt"
OPTIONS_FILE = "/content/drive/MyDrive/Mandarin_answers.txt"

OUTPUT_DIR = "/content/drive/MyDrive/Mandarin_SLA_Project_CSEE"
os.makedirs(OUTPUT_DIR, exist_ok=True)

# ---- UPDATED: LATEST MIXED(30+5 )) MODELS ----
MODEL_DIRS = [
    "/content/drive/MyDrive/MUSE_artifacts/babyroberta_30M_plus5M_seed42_final",
    "/content/drive/MyDrive/MUSE_artifacts/babyroberta_30M_plus5M_seed97_final",
    "/content/drive/MyDrive/MUSE_artifacts/babyroberta_30M_plus5M_seed123_final",
    "/content/drive/MyDrive/MUSE_artifacts/babyroberta_30M_plus5M_seed420_final",
    "/content/drive/MyDrive/MUSE_artifacts/babyroberta_30M_plus5M_seed999_final",
]

OPTIONS = [
    "on", "at", "like", "as", "with", "inside", "of", "among", "in",
    "by", "from", "during", "about", "near", "out", "round", "until",
    "along", "for", "against", "across", "to", "off", "upon", "towards",
    "under", "around", "behind", "besides", "within", "beside", "into",
    "between", "up", "over", "before", "above", "down", "after", "till",
    "toward", "without"
]

# -------------------------------------------------
# DEVICE
# -------------------------------------------------
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

# -------------------------------------------------
# LOAD SENTENCES (ONCE)
# -------------------------------------------------
sentences = []
with open(INPUT_FILE, "r", encoding="utf-8") as f:
    for line in f:
        line = line.strip()
        if not line:
            continue

        line = line.replace("____", "[MASK]")
        line = line.replace(" ___ ", " [MASK] ")
        line = line.replace("___", "[MASK]")
        line = line.replace("__", "[MASK]")

        if "[MASK]" not in line:
            parts = line.split()
            if len(parts) > 1:
                parts.insert(-1, "[MASK]")
                line = " ".join(parts)
            else:
                line = line + " [MASK]"

        sentences.append(line)

print("Loaded", len(sentences), "sentences.")

# -------------------------------------------------
# LOAD OPTION LINES (ONCE)
# -------------------------------------------------
with open(OPTIONS_FILE, "r", encoding="utf-8") as f:
    option_lines = [line.strip() for line in f if line.strip()]

if len(option_lines) != len(sentences):
    raise ValueError(
        f"OPTIONS_FILE line count ({len(option_lines)}) must match INPUT_FILE line count ({len(sentences)})."
    )

# -------------------------------------------------
# LOOP OVER MODELS
# -------------------------------------------------
for model_dir in MODEL_DIRS:
    model_name = os.path.basename(model_dir.rstrip("/"))
    print("\n========================================")
    print("Running model:", model_name)
    print("========================================")

    tokenizer = AutoTokenizer.from_pretrained(model_dir, use_fast=True)
    model = AutoModelForMaskedLM.from_pretrained(model_dir)
    model.to(device)
    model.eval()

    option_token_ids = {w: tokenizer.convert_tokens_to_ids(w) for w in OPTIONS}

    mask_token = tokenizer.mask_token
    mask_token_id = tokenizer.mask_token_id

    def predict_masked_word(sentence):
        sent = sentence.replace("[MASK]", mask_token)

        inputs = tokenizer(sent, return_tensors="pt")
        inputs = {k: v.to(device) for k, v in inputs.items()}

        mask_positions = (inputs["input_ids"] == mask_token_id).nonzero(as_tuple=False)
        if mask_positions.numel() == 0:
            return {w: 0.0 for w in OPTIONS}

        token_idx = int(mask_positions[0, 1].item())

        with torch.no_grad():
            logits = model(**inputs).logits

        probs = torch.softmax(logits[0, token_idx], dim=-1)

        return {
            w: float(probs[option_token_ids[w]].item())
            if option_token_ids[w] is not None else 0.0
            for w in OPTIONS
        }

    # -------------------------------------------------
    # RAW
    # -------------------------------------------------
    rows = []
    for s in sentences:
        row = {"Question": s}
        row.update(predict_masked_word(s))
        rows.append(row)

    df_raw = pd.DataFrame(rows)
    raw_path = f"{OUTPUT_DIR}/{model_name}_raw.csv"
    df_raw.to_csv(raw_path, index=False)
    print("Saved:", raw_path)

    # -------------------------------------------------
    # NORMALIZED
    # -------------------------------------------------
    normalized_rows = []
    for i, allowed in enumerate(option_lines):
        allowed_list = [w.strip() for w in allowed.split(",") if w.strip()]

        total = sum(df_raw.loc[i, w] for w in allowed_list if w in df_raw.columns)

        row = {"sentence": sentences[i]}
        for w in OPTIONS:
            row[w] = df_raw.loc[i, w] / total if (w in allowed_list and total > 0) else 0.0

        normalized_rows.append(row)

    df_norm = pd.DataFrame(normalized_rows)
    norm_path = f"{OUTPUT_DIR}/{model_name}_normalized.csv"
    df_norm.to_csv(norm_path, index=False)
    print("Saved:", norm_path)

    del model
    torch.cuda.empty_cache()

print("\nALL MODELS FINISHED.")

In [ ]:
# ==============================
# Imports + Setup
# ==============================

import os
import random
import numpy as np
import torch
from google.colab import drive
from datasets import load_dataset
from transformers import (
    RobertaTokenizerFast,
    RobertaConfig,
    RobertaForMaskedLM,
    DataCollatorForLanguageModeling,
    TrainingArguments,
    Trainer,
    set_seed
)

# Mount Drive
drive.mount('/content/drive')

In [ ]:
# ==============================
# Seeds + Paths
# ==============================

SEEDS = [42, 97, 123, 420, 999]

CORPUS_25M = "/content/main_25M.txt"        # translated (25M)
CORPUS_5M  = "/content/english_5M.txt"     # original English (5M)

TOKENIZER_DIR = "/content/drive/MyDrive/MUSE_artifacts/babyroberta_tokenizer"
BASE_OUTPUT = "/content/drive/MyDrive/MUSE_artifacts"

In [ ]:
# ==============================
# Seeds + Paths
# ==============================

SEEDS = [42, 97, 123, 420, 999]

CORPUS_25M = "/content/main_25M.txt"        # translated (25M)
CORPUS_5M  = "/content/english_5M.txt"     # original English (5M)

TOKENIZER_DIR = "/content/drive/MyDrive/MUSE_artifacts/babyroberta_tokenizer"
BASE_OUTPUT = "/content/drive/MyDrive/MUSE_artifacts"

In [ ]:
# ==============================
# Load Tokenizer
# ==============================

tokenizer = RobertaTokenizerFast(
    tokenizer_file=f"{TOKENIZER_DIR}/tokenizer.json",
    max_len=512
)

In [ ]:
# ==============================
# Load Datasets
# ==============================

dataset_25M = load_dataset("text", data_files={"train": CORPUS_25M})
dataset_5M  = load_dataset("text", data_files={"train": CORPUS_5M})

In [ ]:
# ==============================
# Tokenization
# ==============================

def tokenize_function(examples):
    return tokenizer(
        examples["text"],
        truncation=True,
        max_length=512,
        return_special_tokens_mask=True,
    )

tokenized_25M = dataset_25M.map(
    tokenize_function,
    batched=True,
    remove_columns=["text"],
    num_proc=2
)

tokenized_5M = dataset_5M.map(
    tokenize_function,
    batched=True,
    remove_columns=["text"],
    num_proc=2
)

In [ ]:
# ==============================
# Group Texts into Blocks
# ==============================

block_size = 512

def group_texts(examples):
    concatenated = {}
    for key in examples.keys():
        concatenated[key] = sum(examples[key], [])

    total_length = len(concatenated["input_ids"])
    total_length = (total_length // block_size) * block_size

    result = {}
    for key in concatenated.keys():
        result[key] = [
            concatenated[key][i : i + block_size]
            for i in range(0, total_length, block_size)
        ]

    return result

lm_25M = tokenized_25M.map(group_texts, batched=True, num_proc=2)
lm_5M  = tokenized_5M.map(group_texts, batched=True, num_proc=2)

In [ ]:
# ==============================
# MLM Data Collator
# ==============================

data_collator = DataCollatorForLanguageModeling(
    tokenizer=tokenizer,
    mlm=True,
    mlm_probability=0.15,
)

In [ ]:
# ==============================
# TRAINING LOOP
# ==============================

for SEED in SEEDS:

    print("\n==============================")
    print(f"Training Seed: {SEED}")
    print("==============================")

    # ------------------------------
    # Set Seed
    # ------------------------------
    random.seed(SEED)
    np.random.seed(SEED)
    torch.manual_seed(SEED)
    torch.cuda.manual_seed_all(SEED)
    set_seed(SEED)

    # ------------------------------
    # Create Model
    # ------------------------------
    config = RobertaConfig(
        vocab_size=32000,
        max_position_embeddings=514,
        num_attention_heads=8,
        num_hidden_layers=6,
        hidden_size=512,
        intermediate_size=2048,
    )

    model = RobertaForMaskedLM(config)

    # ============================
    # STAGE 1 → TRAIN ON 25M
    # ============================
    trainer_25M = Trainer(
        model=model,
        args=TrainingArguments(
            output_dir=f"{BASE_OUTPUT}/seed{SEED}_stage1",
            num_train_epochs=5,
            per_device_train_batch_size=16,
            learning_rate=5e-4,
            weight_decay=0.01,
            warmup_steps=1000,
            logging_steps=500,
            fp16=True,
            report_to="none",
            seed=SEED,
        ),
        train_dataset=lm_25M["train"],
        data_collator=data_collator,
    )

    trainer_25M.train()
    print("Stage 1 done (25M)")

    # ============================
    # PARTIAL FREEZE (IMPORTANT)
    # ============================
    for param in model.roberta.embeddings.parameters():
        param.requires_grad = False

    for param in model.roberta.encoder.layer[:3].parameters():
        param.requires_grad = False

    print("Partial freeze applied")

    # ============================
    # STAGE 2 → FINE-TUNE ON 5M
    # ============================
    trainer_5M = Trainer(
        model=model,
        args=TrainingArguments(
            output_dir=f"{BASE_OUTPUT}/seed{SEED}_stage2",
            num_train_epochs=3,
            per_device_train_batch_size=16,
            learning_rate=2e-4,  # LOWER LR
            weight_decay=0.01,
            logging_steps=500,
            fp16=True,
            report_to="none",
            seed=SEED,
        ),
        train_dataset=lm_5M["train"],
        data_collator=data_collator,
    )

    trainer_5M.train()

    # ------------------------------
    # Save Final Model
    # ------------------------------
    FINAL_DIR = f"{BASE_OUTPUT}/babyroberta_25M_then_5M_freeze_seed{SEED}"
    trainer_5M.save_model(FINAL_DIR)
    tokenizer.save_pretrained(FINAL_DIR)

    print("Saved model:", FINAL_DIR)

print("\n ALL MODELS TRAINED SUCCESSFULLY ")

In [ ]:
# ============================================
# FINAL — PREPOSITION-AWARE + FAST + CORRECT
# ============================================

import torch
import numpy as np
import re
import math
import os
from tqdm import tqdm
from transformers import RobertaForMaskedLM, RobertaTokenizerFast

os.environ["CUDA_LAUNCH_BLOCKING"] = "1"

# --------------------------------------------
# DEVICE
# --------------------------------------------
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

# --------------------------------------------
# TOKENIZER
# --------------------------------------------
TOKENIZER_DIR = "/content/drive/MyDrive/MUSE_artifacts/babyroberta_tokenizer"
tokenizer = RobertaTokenizerFast.from_pretrained(TOKENIZER_DIR)

# --------------------------------------------
# PREPOSITIONS
# --------------------------------------------
PREPOSITIONS = {
    "in","on","at","to","for","with","of","by","from",
    "about","above","across","after","against","along","among","around",
    "as","before","behind","below","beneath","beside","between","beyond",
    "during","except","inside","into","near","off","onto","out","outside",
    "over","past","since","through","throughout","toward","under","underneath",
    "until","up","upon","within","without"
}

# --------------------------------------------
# SENTENCE FILTER (FAIR VERSION)
# --------------------------------------------
def extract_sentences_with_prepositions(text):
    sentences = re.split(r'[.!?]', text)

    preposition_sentences = []
    fallback_sentences = []

    for sent in sentences:
        words = re.findall(r'\b\w+\b', sent.lower())

        if len(words) > 5:
            fallback_sentences.append(sent.strip())

            if any(w in PREPOSITIONS for w in words):
                preposition_sentences.append(sent.strip())

    # ✅ fairness fix
    if len(preposition_sentences) > 0:
        return preposition_sentences
    else:
        return fallback_sentences


# --------------------------------------------
# 🚀 FIXED PSEUDO PERPLEXITY
# --------------------------------------------
def compute_pseudo_perplexity(text, model, tokenizer, max_length=128, batch_size=32):

    enc = tokenizer(
        text,
        return_tensors="pt",
        truncation=True,
        max_length=max_length
    )

    input_ids = enc["input_ids"][0]
    attention_mask = enc["attention_mask"][0]

    seq_len = input_ids.size(0)

    if seq_len < 3:
        return None

    masked_inputs = []
    target_tokens = []

    # mask each token (except special tokens)
    for i in range(1, seq_len - 1):
        masked = input_ids.clone()
        masked[i] = tokenizer.mask_token_id

        masked_inputs.append(masked)
        target_tokens.append(input_ids[i].item())

    masked_inputs = torch.stack(masked_inputs)
    attention_mask = attention_mask.unsqueeze(0).repeat(len(masked_inputs), 1)

    masked_inputs = masked_inputs.to(device)
    attention_mask = attention_mask.to(device)

    log_likelihood = 0.0
    token_count = 0

    # batching
    for batch_start in range(0, len(masked_inputs), batch_size):

        batch_input = masked_inputs[batch_start:batch_start+batch_size]
        batch_mask = attention_mask[batch_start:batch_start+batch_size]

        with torch.no_grad():
            outputs = model(batch_input, attention_mask=batch_mask)
            logits = outputs.logits

        for j in range(batch_input.size(0)):

            original_idx = batch_start + j
            token_position = original_idx + 1

            true_token = target_tokens[original_idx]

            if true_token >= logits.size(-1):
                continue

            log_probs = torch.log_softmax(logits[j, token_position], dim=-1)

            log_likelihood += log_probs[true_token].item()
            token_count += 1

    if token_count == 0:
        return None

    return math.exp(-log_likelihood / token_count)


# --------------------------------------------
# DATA LOADER
# --------------------------------------------
def load_essays(path, group):
    with open(path, "r", encoding="utf-8") as f:
        text = f.read()

    essays = text.split("\n")
    essays = [e.strip() for e in essays if len(e.split()) > 20]

    if len(essays) < 50:
        words = text.split()
        chunk_size = 150

        essays = [
            " ".join(words[i:i+chunk_size])
            for i in range(0, len(words), chunk_size)
        ]

    essays = essays[:200]

    print(f"{group} essays:", len(essays))
    return essays


# --------------------------------------------
# MODELS
# --------------------------------------------
models_config = {
    "Seed42_freeze": "/content/drive/MyDrive/MUSE_artifacts/babyroberta_25M_then_5M_freeze_seed42",
    "Seed97_freeze": "/content/drive/MyDrive/MUSE_artifacts/babyroberta_25M_then_5M_freeze_seed97",
    "Seed123_freeze": "/content/drive/MyDrive/MUSE_artifacts/babyroberta_25M_then_5M_freeze_seed123",
    "Seed420_freeze": "/content/drive/MyDrive/MUSE_artifacts/babyroberta_25M_then_5M_freeze_seed420",
    "Seed999_freeze": "/content/drive/MyDrive/MUSE_artifacts/babyroberta_25M_then_5M_freeze_seed999",

}

# --------------------------------------------
# DATASETS
# --------------------------------------------
essay_files = {
    "Native": "/content/drive/MyDrive/SLA_Project/ens_icnale_clean.txt",
    "Chinese": "/content/drive/MyDrive/SLA_Project/chinese_icnale_clean.txt",
    "Japanese": "/content/drive/MyDrive/SLA_Project/japanese_icnale_clean.txt",
    "Korean": "/content/drive/MyDrive/SLA_Project/kor_icnale_clean.txt",
    "Thai": "/content/drive/MyDrive/SLA_Project/tha_icnale_clean.txt",
    "Filipino": "/content/drive/MyDrive/SLA_Project/phl_icnale_clean.txt",
    "Urdu": "/content/drive/MyDrive/SLA_Project/urdu_icnale_clean.txt",
}

all_results = {}

# ============================================
# MAIN LOOP
# ============================================
for model_name, model_path in models_config.items():

    print("\n========================================")
    print("Loading:", model_name)
    print("========================================")

    torch.cuda.empty_cache()

    model = RobertaForMaskedLM.from_pretrained(model_path).to(device)
    model.eval()

    results = {}

    for group, path in essay_files.items():

        if not os.path.exists(path):
            print(f"⚠️ Missing: {group}")
            continue

        print(f"\nEvaluating {group}...")

        essays = load_essays(path, group)
        ppl_scores = []

        for essay in tqdm(essays):

            sentences = extract_sentences_with_prepositions(essay)

            if len(sentences) == 0:
                sentences = [essay]

            for sent in sentences:
                ppl = compute_pseudo_perplexity(sent, model, tokenizer)

                if ppl is not None:
                    ppl_scores.append(ppl)

        avg_ppl = np.mean(ppl_scores) if len(ppl_scores) > 0 else float("nan")
        results[group] = avg_ppl

        print(f"{group}: {avg_ppl:.3f}")

    all_results[model_name] = results


# ============================================
# FINAL TABLE
# ============================================
print("\n\nFINAL COMPARISON TABLE")
print("========================================")

groups = list(essay_files.keys())

print(f"{'Model':<30}", end="")
for g in groups:
    print(f"{g:<12}", end="")
print()

for model_name in all_results.keys():
    print(f"{model_name:<30}", end="")

    for g in groups:
        val = all_results[model_name].get(g, float("nan"))

        if np.isnan(val):
            print(f"{'NaN':<12}", end="")
        else:
            print(f"{val:.2f}".ljust(12), end="")

    print()

In [ ]:
import numpy as np

# =========================
# Store results (your data)
# =========================
data = {
    "Native":    [52.51, 63.28, 75.73, 58.17, 54.53],
    "Chinese":   [66.24, 79.99, 92.59, 70.25, 71.02],
    "Japanese":  [50.81, 58.64, 71.77, 54.33, 50.27],
    "Korean":    [70.67, 85.77, 99.31, 75.22, 78.63],
    "Thai":      [46.10, 52.06, 64.55, 49.02, 44.78],
    "Filipino":  [46.96, 53.61, 67.74, 50.36, 46.69],
    "Urdu":      [57.66, 65.71, 80.52, 60.48, 62.30]
}

# =========================
# EXCLUDE languages if needed
# =========================
exclude = ["Korean", "Urdu"]   # <-- remove these
filtered_data = {k: v for k, v in data.items() if k not in exclude}

# =========================
# Compute averages
# =========================
averages = {}

for lang, values in filtered_data.items():
    avg = np.mean(values)
    averages[lang] = round(avg, 2)

# =========================
# Print results
# =========================
print("\nAVERAGE RESULTS (Freeze Model):\n")
for lang, avg in averages.items():
    print(f"{lang}: {avg}")

In [ ]:
# Full corrected ONE-CELL code (RoBERTa-safe) — writes separate CSVs per model

import os
import torch
import pandas as pd
from transformers import AutoTokenizer, AutoModelForMaskedLM

# -------------------------------------------------
# CONFIG
# -------------------------------------------------
INPUT_FILE = "/content/drive/MyDrive/Mandarin_Question_MASK.txt"
OPTIONS_FILE = "/content/drive/MyDrive/Mandarin_answers.txt"

OUTPUT_DIR = "/content/drive/MyDrive/Mandarin_SLA_Project_CSEE"
os.makedirs(OUTPUT_DIR, exist_ok=True)

# ---- UPDATED: LATEST MIXED(30+5 )) MODELS ----
MODEL_DIRS = [
    "/content/drive/MyDrive/MUSE_artifacts/babyroberta_25M_then_5M_freeze_seed42",
    "/content/drive/MyDrive/MUSE_artifacts/babyroberta_25M_then_5M_freeze_seed97",
    "/content/drive/MyDrive/MUSE_artifacts/babyroberta_25M_then_5M_freeze_seed123",
    "/content/drive/MyDrive/MUSE_artifacts/babyroberta_25M_then_5M_freeze_seed420",
    "/content/drive/MyDrive/MUSE_artifacts/babyroberta_25M_then_5M_freeze_seed999",
]

OPTIONS = [
    "on", "at", "like", "as", "with", "inside", "of", "among", "in",
    "by", "from", "during", "about", "near", "out", "round", "until",
    "along", "for", "against", "across", "to", "off", "upon", "towards",
    "under", "around", "behind", "besides", "within", "beside", "into",
    "between", "up", "over", "before", "above", "down", "after", "till",
    "toward", "without"
]

# -------------------------------------------------
# DEVICE
# -------------------------------------------------
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

# -------------------------------------------------
# LOAD SENTENCES (ONCE)
# -------------------------------------------------
sentences = []
with open(INPUT_FILE, "r", encoding="utf-8") as f:
    for line in f:
        line = line.strip()
        if not line:
            continue

        line = line.replace("____", "[MASK]")
        line = line.replace(" ___ ", " [MASK] ")
        line = line.replace("___", "[MASK]")
        line = line.replace("__", "[MASK]")

        if "[MASK]" not in line:
            parts = line.split()
            if len(parts) > 1:
                parts.insert(-1, "[MASK]")
                line = " ".join(parts)
            else:
                line = line + " [MASK]"

        sentences.append(line)

print("Loaded", len(sentences), "sentences.")

# -------------------------------------------------
# LOAD OPTION LINES (ONCE)
# -------------------------------------------------
with open(OPTIONS_FILE, "r", encoding="utf-8") as f:
    option_lines = [line.strip() for line in f if line.strip()]

if len(option_lines) != len(sentences):
    raise ValueError(
        f"OPTIONS_FILE line count ({len(option_lines)}) must match INPUT_FILE line count ({len(sentences)})."
    )

# -------------------------------------------------
# LOOP OVER MODELS
# -------------------------------------------------
for model_dir in MODEL_DIRS:
    model_name = os.path.basename(model_dir.rstrip("/"))
    print("\n========================================")
    print("Running model:", model_name)
    print("========================================")

    tokenizer = AutoTokenizer.from_pretrained(model_dir, use_fast=True)
    model = AutoModelForMaskedLM.from_pretrained(model_dir)
    model.to(device)
    model.eval()

    option_token_ids = {w: tokenizer.convert_tokens_to_ids(w) for w in OPTIONS}

    mask_token = tokenizer.mask_token
    mask_token_id = tokenizer.mask_token_id

    def predict_masked_word(sentence):
        sent = sentence.replace("[MASK]", mask_token)

        inputs = tokenizer(sent, return_tensors="pt")
        inputs = {k: v.to(device) for k, v in inputs.items()}

        mask_positions = (inputs["input_ids"] == mask_token_id).nonzero(as_tuple=False)
        if mask_positions.numel() == 0:
            return {w: 0.0 for w in OPTIONS}

        token_idx = int(mask_positions[0, 1].item())

        with torch.no_grad():
            logits = model(**inputs).logits

        probs = torch.softmax(logits[0, token_idx], dim=-1)

        return {
            w: float(probs[option_token_ids[w]].item())
            if option_token_ids[w] is not None else 0.0
            for w in OPTIONS
        }

    # -------------------------------------------------
    # RAW
    # -------------------------------------------------
    rows = []
    for s in sentences:
        row = {"Question": s}
        row.update(predict_masked_word(s))
        rows.append(row)

    df_raw = pd.DataFrame(rows)
    raw_path = f"{OUTPUT_DIR}/{model_name}_raw.csv"
    df_raw.to_csv(raw_path, index=False)
    print("Saved:", raw_path)

    # -------------------------------------------------
    # NORMALIZED
    # -------------------------------------------------
    normalized_rows = []
    for i, allowed in enumerate(option_lines):
        allowed_list = [w.strip() for w in allowed.split(",") if w.strip()]

        total = sum(df_raw.loc[i, w] for w in allowed_list if w in df_raw.columns)

        row = {"sentence": sentences[i]}
        for w in OPTIONS:
            row[w] = df_raw.loc[i, w] / total if (w in allowed_list and total > 0) else 0.0

        normalized_rows.append(row)

    df_norm = pd.DataFrame(normalized_rows)
    norm_path = f"{OUTPUT_DIR}/{model_name}_normalized.csv"
    df_norm.to_csv(norm_path, index=False)
    print("Saved:", norm_path)

    del model
    torch.cuda.empty_cache()

print("\nALL MODELS FINISHED.")